## Attempt to compile the whole dataset

In [ ]:
# Smart, no-touch recompilation of Solidity projects under ./contracts
# - NEVER edits Solidity sources
# - Prefers ./contracts/<addr>/standard_input.json (compile_standard)
# - Else compiles all .sol in the project (compile_standard); last resort: single main file
# - Version strategy:
#     1) Use CSV 'compiler_version' if reasonable
#     2) Else choose highest version allowed by pragmas across files
#     3) If legacy unnamed fallback detected, prefer 0.5.17 (if pragma allows or no pragma)
#     4) On failures:
#        - If "requires different compiler version" → retry with pragma-best
#        - If "Invalid SPDX license identifier" → try older minors (0.8→0.7→0.6→0.5→0.4)
#        - If "Invalid EVM version requested" → retry without evmVersion in settings
#        - If type error like contract vs address(0) in 0.5.x+ → report "needs source change"
#
# Outputs:
#   compilation_results_final.csv (per-address outcome)

import json, re, time, traceback
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import pandas as pd
from solcx import (
    install_solc,
    set_solc_version,
    compile_source,
    compile_standard,
    get_installed_solc_versions,
)
from solcx.exceptions import SolcError

try:
    from tqdm import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

# ---------------- config ----------------
INPUT_CSV     = "disl_subset.csv"   # change if needed
PROJECTS_DIR  = Path("contracts")
OUT_RECOMPILE = "compilation_results_final.csv"
RECOMPILE_ALL = True
SLEEP_BETWEEN = 0.0

# ---------------- version helpers ----------------
SEMVER3   = re.compile(r"(\d+)\.(\d+)\.(\d+)")
PRAGMA    = re.compile(r"pragma\s+solidity\s+([^;]+);", re.IGNORECASE)
OP_VER    = re.compile(r"([~^]|>=|<=|>|<)?\s*(\d+)\.(\d+)\.(\d+)")

LATEST_PATCH = {(0,4):26,(0,5):17,(0,6):12,(0,7):6,(0,8):30}
MINOR_ORDER  = [(0,8),(0,7),(0,6),(0,5),(0,4)]
VALID_MINORS = {4,5,6,7,8}

LEGACY_FALLBACK_RE = re.compile(r"function\s*\(\)\s*(external|public)", re.IGNORECASE)

def vtup(s: str):
    m = SEMVER3.search(s)
    return tuple(map(int, m.groups())) if m else (0,0,0)

def vstr(t): return f"{t[0]}.{t[1]}.{t[2]}"

def latest_in_minor(M,m): return (M,m, LATEST_PATCH.get((M,m), 0))

def parse_pragma_expr(expr: str):
    lo,hi = (0,0,0),(99,99,99)  # half-open [lo, hi)
    toks = OP_VER.findall(expr)
    if not toks: return lo,hi
    def vmax(a,b): return a if a>b else b
    def vmin(a,b): return a if a<b else b
    for op,a,b,c in toks:
        vt = (int(a),int(b),int(c))
        if op in ("",None): lo=vmax(lo,vt); hi=vmin(hi,(vt[0],vt[1],vt[2]+1))
        elif op=="^": lo=vmax(lo,vt); hi=vmin(hi,(vt[0],vt[1]+1,0))
        elif op=="~": lo=vmax(lo,vt); hi=vmin(hi,(vt[0],vt[1]+1,0))
        elif op==">=": lo=vmax(lo,vt)
        elif op==">":  lo=vmax(lo,(vt[0],vt[1],vt[2]+1))
        elif op=="<=": hi=vmin(hi,(vt[0],vt[1],vt[2]+1))
        elif op=="<":  hi=vmin(hi,vt)
    return lo,hi

def scan_pragmas(sources: Dict[str,str]) -> Tuple[Tuple[int,int,int],Tuple[int,int,int],bool]:
    have=False
    lo,hi = (0,0,0),(99,99,99)
    legacy=False
    for content in sources.values():
        if LEGACY_FALLBACK_RE.search(content or ""):
            legacy=True
        for pm in PRAGMA.finditer(content or ""):
            have=True
            a,b = parse_pragma_expr(pm.group(1))
            # intersect
            lo = a if a>lo else lo
            hi = b if b<hi else hi
    return lo,hi,legacy

def is_reasonable_csv_version(v: str) -> bool:
    m = SEMVER3.search(v or "")
    if not m: return False
    M,mn,p = map(int, m.groups())
    if M!=0 or mn not in VALID_MINORS: return False
    return 0 <= p <= LATEST_PATCH.get((M,mn), -1)

def pick_initial_version(csv_version: str, sources: Dict[str,str]) -> str:
    if isinstance(csv_version,str) and csv_version.strip() and is_reasonable_csv_version(csv_version):
        return ".".join(SEMVER3.search(csv_version).groups())
    lo,hi,legacy = scan_pragmas(sources)
    if hi != (99,99,99):  # have pragma info
        # choose the latest patch strictly below hi and >= lo
        cand = (hi[0],hi[1],max(0,hi[2]-1))
        if cand < lo:
            cand = latest_in_minor(lo[0], lo[1])
        return vstr(cand)
    return "0.8.21"  # default

def fallback_candidates(base_version: str, lo_hi_legacy: Tuple[Tuple[int,int,int],Tuple[int,int,int],bool]) -> List[str]:
    lo,hi,legacy = lo_hi_legacy
    candidates = []
    # 0) start with the chosen one
    candidates.append(base_version)
    # 1) pragma-best (highest inside range)
    if hi != (99,99,99):
        best = (hi[0],hi[1],max(0,hi[2]-1))
        if vstr(best) not in candidates:
            candidates.append(vstr(best))
    # 2) if legacy unnamed fallback present, prefer 0.5.17 (if allowed by pragma or no pragma)
    if legacy:
        if hi == (99,99,99) or (0,5,17) < hi and (0,5,0) >= lo:
            if "0.5.17" not in candidates:
                candidates.append("0.5.17")
    # 3) patch-down within same minor
    M,m,_ = vtup(base_version)
    if (M,m) in LATEST_PATCH:
        for p in range(LATEST_PATCH[(M,m)], -1, -1):
            vv = f"{M}.{m}.{p}"
            if vv not in candidates: candidates.append(vv)
    # 4) earlier minors (latest patch per minor)
    for M2,m2 in MINOR_ORDER:
        vv = vstr(latest_in_minor(M2,m2))
        if vv not in candidates:
            candidates.append(vv)
    return candidates

# ---------------- IO helpers (no source edits) ----------------
_SEMVER_RE  = re.compile(r"(\d+)\.(\d+)\.(\d+)")
MAX_SOLC = "0.8.30"   # clamp bogus >0.8.x picks

def _semver_tuple(v: str):
    m = _SEMVER_RE.search(v or "")
    return tuple(map(int, m.groups())) if m else (0,0,0)

def _feature_floor(sources: Dict[str, str]) -> str:
    """Return the minimum compiler version implied by language features in the code."""
    blob = "\n".join(sources.values())
    floor = (0,4,0)

    # Solidity features → minimum versions (conservative)
    if re.search(r"\berror\s+[A-Za-z_]\w*\s*\(", blob):               # custom errors
        floor = max(floor, (0,8,4))
    if re.search(r"\babstract\s+contract\b", blob) or \
       re.search(r"\boverride\b", blob) or \
       re.search(r"\bvirtual\b", blob) or \
       re.search(r"\breceive\s*\(\)\s*external", blob) or \
       re.search(r"\btry\b\s+[A-Za-z_]\w*\s*\(", blob) or \
       re.search(r"\bcatch\b", blob):                                  # 0.6.x family
        floor = max(floor, (0,6,0))
    if re.search(r"\bimmutable\b", blob):                              # immutable
        floor = max(floor, (0,6,5))
    if re.search(r"\bcalldata\b", blob):                               # 0.5.x grammar
        floor = max(floor, (0,5,0))

    return ".".join(map(str, floor))

def _clamp_max(v: str, max_v: str = MAX_SOLC) -> str:
    return max_v if _semver_tuple(v) > _semver_tuple(max_v) else v

def _collect_sources_from_dir(project_dir: Path) -> Dict[str, str]:
    sources: Dict[str, str] = {}
    for p in project_dir.rglob("*.sol"):
        rel = p.relative_to(project_dir).as_posix()
        try:
            txt = p.read_text()
        except Exception:
            txt = ""
        sources[rel] = txt
    return sources

def _load_standard_input(project_dir: Path) -> Optional[dict]:
    f = project_dir / "standard_input.json"
    if not f.exists(): return None
    try:
        raw = json.loads(f.read_text())
        return raw if isinstance(raw, dict) else None
    except Exception:
        return None

# ---------------- compile plumbing ----------------
_current_solc: Optional[str] = None

def _ensure_solc(version: str) -> Optional[str]:
    global _current_solc
    try:
        if _current_solc == version: return None
        installed = {str(v) for v in get_installed_solc_versions()}
        if version not in installed: install_solc(version)
        set_solc_version(version)
        _current_solc = version
        return None
    except Exception as e:
        return f"Failed to set solc {version}: {e}"

def _compile_standard_resilient(payload: dict, version: str):
    try:
        return compile_standard(payload)
    except SolcError as e1:
        text = (getattr(e1,"stderr_data","") or "") + (getattr(e1,"stdout_data","") or "") + str(e1)
        if "Invalid EVM version requested" in text:
            # Retry without evmVersion
            st = dict(payload.get("settings") or {})
            st.pop("evmVersion", None)
            p2 = dict(payload); p2["settings"]=st
            return compile_standard(p2)
        raise

def _err_text(e: Exception) -> str:
    if isinstance(e, SolcError):
        stderr = getattr(e, "stderr_data", None) or getattr(e, "stderr", None) or ""
        stdout = getattr(e, "stdout_data", None) or getattr(e, "stdout", None) or ""
        tail = lambda s: s[-1200:] if isinstance(s,str) else ""
        return f"--- stderr (tail) ---\n{tail(stderr)}\n--- stdout (tail) ---\n{tail(stdout)}"
    return str(e)

def _looks_spdx_error(e: Exception) -> bool:
    txt = _err_text(e)
    return "Invalid SPDX license identifier" in txt

def _looks_version_mismatch(e: Exception) -> bool:
    txt = _err_text(e)
    return "requires different compiler version" in txt or "Source file requires different compiler version" in txt

def _looks_contract_vs_address0(e: Exception) -> bool:
    txt = _err_text(e)
    return "Operator !=" in txt and "contract" in txt and "address" in txt

def _compile_once(version: str, std: Optional[dict], sources: Optional[Dict[str,str]]) -> Tuple[bool,str,str]:
    err = _ensure_solc(version)
    if err: return False, err, "init"
    if std and isinstance(std.get("sources"), dict) and std["sources"]:
        try:
            settings = dict(std.get("settings") or {})
            if "optimizer" not in settings:
                settings["optimizer"] = {"enabled": True, "runs": 200}
            payload = {"language":"Solidity","sources":std["sources"],"settings":settings}
            _ = _compile_standard_resilient(payload, version)
            return True, "Compiled OK", "standard"
        except Exception as e:
            return False, f"SolcError (v{version}, mode=standard)\n{_err_text(e)}", "standard"
    if sources:
        try:
            payload={"language":"Solidity","sources":{k:{"content":v} for k,v in sources.items()},
                     "settings":{"optimizer":{"enabled":True,"runs":200}}}
            _ = _compile_standard_resilient(payload, version)
            return True, "Compiled OK", "all_files"
        except Exception as e_all:
            try:
                main_rel = max(sources.items(), key=lambda kv: len(kv[1]))[0]
                _ = compile_source(sources[main_rel])
                return True, f"Compiled OK (single file: {main_rel})", "single_file"
            except Exception as e_single:
                joined = f"SolcError (v{version}, mode=all_files)\n{_err_text(e_all)}\n---\nSolcError (v{version}, mode=single_file)\n{_err_text(e_single)}"
                return False, joined, "all_files"
    return False, "No .sol sources found", "none"

def compile_project(project_dir: Path, csv_version: str):
    std = _load_standard_input(project_dir)
    sources = None if std else _collect_sources_from_dir(project_dir)
    n_sources = (len(std["sources"]) if std and "sources" in std else (len(sources) if sources else 0))
    src_for_infer = {k:(v.get("content") if isinstance(v, dict) else str(v)) for k,v in (std["sources"] if std else (sources or {})).items()} if (std or sources) else {}

    lo,hi,legacy = scan_pragmas(src_for_infer)
    initial = pick_initial_version(csv_version, src_for_infer)
    cands = fallback_candidates(initial, (lo,hi,legacy))

    # Try candidates; add special handling for SPDX / version mismatch
    spdx_downgraded = False
    tried = set()
    for ver in cands:
        if ver in tried: continue
        tried.add(ver)

        ok, msg, mode = _compile_once(ver, std, sources)
        if ok:
            tag = "initial" if ver == initial else ("pragma_best" if ver == vstr((hi[0],hi[1],max(0,hi[2]-1))) else ("spdx_downgrade" if spdx_downgraded else "fallback"))
            return True, f"Compiled OK ({tag} v{ver}, mode={mode})", ver, n_sources, mode

        # If version mismatch, and we have pragma info, jump directly to pragma-best once
        if _looks_version_mismatch(msg) and hi != (99,99,99):
            pb = vstr((hi[0],hi[1],max(0,hi[2]-1)))
            if pb not in tried:
                ok2, msg2, mode2 = _compile_once(pb, std, sources)
                tried.add(pb)
                if ok2:
                    return True, f"Compiled OK (pragma_best v{pb}, mode={mode2})", pb, n_sources, mode2
                msg = msg + "\n---\n" + msg2  # keep diagnostics

        # If SPDX error, attempt minor downgrade ladder (0.8→0.7→0.6→0.5→0.4)
        if _looks_spdx_error(msg):
            for M2,m2 in MINOR_ORDER:
                vv = vstr(latest_in_minor(M2,m2))
                if vv in tried: continue
                spdx_downgraded = True
                ok3, msg3, mode3 = _compile_once(vv, std, sources)
                tried.add(vv)
                if ok3:
                    return True, f"Compiled OK (spdx_downgrade v{vv}, mode={mode3})", vv, n_sources, mode3
                msg = msg + "\n---\n" + msg3

        # If fundamental type issue (e.g., contract vs address(0)), declare needs-source-change
        if _looks_contract_vs_address0(msg):
            return False, "TypeError: contract vs address(0) comparison under 0.5+ — requires source change (not auto-fixable by compiler version).", initial, n_sources, "none"

    # Exhausted candidates
    return False, (msg if isinstance(msg,str) else "Compilation failed (no further detail)"), initial, n_sources, "none"

# ---------------- main ------------------
df = pd.read_csv(INPUT_CSV)
todo = df.copy() if RECOMPILE_ALL else df.loc[df.get("compile_ok").astype(str).str.lower() != "true"].copy()

rows=[]
it = range(len(todo))
if _HAS_TQDM:
    it = tqdm(it, desc="Recompiling (no source edits)", unit="addr")

for i in it:
    row = todo.iloc[i]
    addr = str(row["contract_address"]).strip().lower()
    proj = PROJECTS_DIR / addr
    t0 = time.perf_counter() if hasattr(time, "perf_counter") else time.time()

    if not proj.exists():
        rows.append({
            "contract_address": addr, "project_dir": str(proj),
            "recompile_ok": False, "recompile_message": "Project directory not found",
            "used_version": "", "n_sources": 0, "mode": "none",
            "elapsed_s": round((time.perf_counter() if hasattr(time, "perf_counter") else time.time()) - t0, 3),
            "original_idx": row.get("original_idx"), "label": row.get("label"),
        })
        continue

    csv_ver = str(row.get("compiler_version") or "").strip()
    ok, msg, used, nsrc, mode = compile_project(proj, csv_ver)

    rows.append({
        "contract_address": addr, "project_dir": str(proj),
        "recompile_ok": bool(ok), "recompile_message": str(msg),
        "used_version": used, "n_sources": int(nsrc), "mode": mode,
        "elapsed_s": round((time.perf_counter() if hasattr(time, "perf_counter") else time.time()) - t0, 3),
        "original_idx": row.get("original_idx"), "label": row.get("label"),
    })
    if SLEEP_BETWEEN>0: time.sleep(SLEEP_BETWEEN)

out = pd.DataFrame(rows)
out.to_csv(OUT_RECOMPILE, index=False)
print(f"[OK] Wrote: {OUT_RECOMPILE} ({len(out)} rows)")

## Compiling the failed ones

In [ ]:
# for the following list, let's fetch the indicies of the rows in "compilation_results_final.csv" where the 'recompile_ok' is not True.
RETRY_ORIGINAL_IDX = df.loc[~df["recompile_ok"], "original_idx"].tolist()
print(RETRY_ORIGINAL_IDX)


In [ ]:
# Recompile ONLY specific rows (by original_idx) in compilation_results_final.csv.
# - Never edits Solidity sources
# - Updates ONLY the selected rows in-place (no new rows/columns)
# - Uses resilient version picking (CSV/prev used -> pragma/range -> feature floor -> error-hinted)
#
# HOW TO USE:
#   1) Set CSV_PATH and PROJECTS_DIR if needed.
#   2) Put the ~100 indices into RETRY_ORIGINAL_IDX (values from the 'original_idx' column).
#   3) Run the cell.

import re, json, time
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import pandas as pd
from solcx import (
    install_solc,
    set_solc_version,
    compile_source,
    compile_standard,
    get_installed_solc_versions,
)
from solcx.exceptions import SolcError

# ---------------- user config ----------------
CSV_PATH        = "compilation_results_final.csv"   # path to your *existing* results CSV
PROJECTS_DIR    = Path("contracts")                 # base folder that holds per-address sources

# ---------------- helpers (no source edits) ----------------
SEMVER3   = re.compile(r"(\d+)\.(\d+)\.(\d+)")
PRAGMA    = re.compile(r"pragma\s+solidity\s+([^;]+);", re.IGNORECASE)
OP_VER    = re.compile(r"([~^]|>=|<=|>|<)?\s*(\d+)\.(\d+)\.(\d+)")
ERR_PRAGMA_SEMVER = re.compile(r"pragma\s+solidity[^0-9]*?(\d+\.\d+\.\d+)", re.IGNORECASE)

LATEST_PATCH = {(0,4):26,(0,5):17,(0,6):12,(0,7):6,(0,8):30}
MINOR_ORDER  = [(0,8),(0,7),(0,6),(0,5),(0,4)]
MAX_SOLC     = "0.8.30"

LEGACY_FALLBACK_RE = re.compile(r"function\s*\(\)\s*(external|public)", re.IGNORECASE)

def vtup(s: str):
    m = SEMVER3.search(s or "")
    return tuple(map(int, m.groups())) if m else (0,0,0)

def vstr(t): return f"{t[0]}.{t[1]}.{t[2]}"

def latest_in_minor(M,m): return (M,m, LATEST_PATCH.get((M,m), 0))

def parse_pragma_expr(expr: str):
    lo,hi = (0,0,0),(99,99,99)  # half-open [lo,hi)
    toks = OP_VER.findall(expr)
    if not toks: return lo,hi
    vmax = lambda a,b: a if a>b else b
    vmin = lambda a,b: a if a<b else b
    for op,a,b,c in toks:
        vt = (int(a),int(b),int(c))
        if op in ("",None): lo=vmax(lo,vt); hi=vmin(hi,(vt[0],vt[1],vt[2]+1))
        elif op=="^": lo=vmax(lo,vt); hi=vmin(hi,(vt[0],vt[1]+1,0))
        elif op=="~": lo=vmax(lo,vt); hi=vmin(hi,(vt[0],vt[1]+1,0))
        elif op==">=": lo=vmax(lo,vt)
        elif op==">":  lo=vmax(lo,(vt[0],vt[1],vt[2]+1))
        elif op=="<=": hi=vmin(hi,(vt[0],vt[1],vt[2]+1))
        elif op=="<":  hi=vmin(hi,vt)
    return lo,hi

def scan_pragmas(sources: Dict[str,str]) -> Tuple[Tuple[int,int,int],Tuple[int,int,int],bool]:
    lo,hi = (0,0,0),(99,99,99)
    legacy=False
    for content in sources.values():
        if LEGACY_FALLBACK_RE.search(content or ""):
            legacy=True
        for pm in PRAGMA.finditer(content or ""):
            a,b = parse_pragma_expr(pm.group(1))
            lo = a if a>lo else lo
            hi = b if b<hi else hi
    return lo,hi,legacy

def _semver_tuple(v: str):
    m = SEMVER3.search(v or "")
    return tuple(map(int, m.groups())) if m else (0,0,0)

def _clamp_max(v: str, max_v: str = MAX_SOLC) -> str:
    return max_v if _semver_tuple(v) > _semver_tuple(max_v) else v

def _feature_floor(sources: Dict[str, str]) -> str:
    """Minimum compiler implied by features (conservative)."""
    blob = "\n".join(sources.values())
    floor = (0,4,0)
    if re.search(r"\bcalldata\b", blob): floor = max(floor, (0,5,0))
    if re.search(r"\bimmutable\b", blob): floor = max(floor, (0,6,5))
    if re.search(r"\babstract\s+contract\b", blob) or re.search(r"\boverride\b", blob) or \
       re.search(r"\bvirtual\b", blob) or re.search(r"\breceive\s*\(\)\s*external", blob) or \
       re.search(r"\btry\b\s+[A-Za-z_]\w*\s*\(", blob) or re.search(r"\bcatch\b", blob):
        floor = max(floor, (0,6,0))
    if re.search(r"\berror\s+[A-Za-z_]\w*\s*\(", blob): floor = max(floor, (0,8,4))
    return ".".join(map(str, floor))

def _collect_sources_from_dir(project_dir: Path) -> Dict[str, str]:
    sources: Dict[str, str] = {}
    for p in project_dir.rglob("*.sol"):
        rel = p.relative_to(project_dir).as_posix()
        try: txt = p.read_text()
        except Exception: txt = ""
        sources[rel] = txt
    return sources

def _load_standard_input(project_dir: Path) -> Optional[dict]:
    f = project_dir / "standard_input.json"
    if not f.exists(): return None
    try:
        raw = json.loads(f.read_text())
        return raw if isinstance(raw, dict) else None
    except Exception:
        return None

_current_solc: Optional[str] = None
def _ensure_solc(version: str) -> Optional[str]:
    global _current_solc
    try:
        if _current_solc == version: return None
        installed = {str(v) for v in get_installed_solc_versions()}
        if version not in installed: install_solc(version)
        set_solc_version(version)
        _current_solc = version
        return None
    except Exception as e:
        return f"Failed to set solc {version}: {e}"

def _err_text(e: Exception) -> str:
    if isinstance(e, SolcError):
        stderr = getattr(e, "stderr_data", None) or getattr(e, "stderr", None) or ""
        stdout = getattr(e, "stdout_data", None) or getattr(e, "stdout", None) or ""
        tail = lambda s: s[-1200:] if isinstance(s,str) else ""
        return f"--- stderr (tail) ---\n{tail(stderr)}\n--- stdout (tail) ---\n{tail(stdout)}"
    return str(e)

def _looks_spdx_error(e_or_msg) -> bool:
    txt = e_or_msg if isinstance(e_or_msg, str) else _err_text(e_or_msg)
    return "Invalid SPDX license identifier" in txt

def _looks_version_mismatch(e_or_msg) -> bool:
    txt = e_or_msg if isinstance(e_or_msg, str) else _err_text(e_or_msg)
    return ("requires different compiler version" in txt) or ("Source file requires different compiler version" in txt)

def _extract_semver_from_error(e_or_msg) -> Optional[str]:
    txt = e_or_msg if isinstance(e_or_msg, str) else _err_text(e_or_msg)
    m = ERR_PRAGMA_SEMVER.search(txt)
    if m:
        v = m.group(1)
        # guard against 0.9.x claims
        return _clamp_max(v)
    return None

def _compile_standard_resilient(payload: dict):
    try:
        return compile_standard(payload)
    except SolcError as e1:
        txt = _err_text(e1)
        if "Invalid EVM version requested" in txt:
            st = dict(payload.get("settings") or {})
            st.pop("evmVersion", None)
            p2 = dict(payload); p2["settings"] = st
            return compile_standard(p2)
        raise

def _compile_once(version: str, std: Optional[dict], sources: Optional[Dict[str,str]]) -> Tuple[bool,str,str]:
    err = _ensure_solc(version)
    if err: return False, err, "init"
    if std and isinstance(std.get("sources"), dict) and std["sources"]:
        try:
            settings = dict(std.get("settings") or {})
            if "optimizer" not in settings:
                settings["optimizer"] = {"enabled": True, "runs": 200}
            payload = {"language":"Solidity",
                       "sources":{k:({"content": (v.get("content") if isinstance(v, dict) else str(v))})
                                  for k,v in std["sources"].items()},
                       "settings":settings}
            _ = _compile_standard_resilient(payload)
            return True, "Compiled OK", "standard"
        except Exception as e:
            return False, f"SolcError (v{version}, mode=standard)\n{_err_text(e)}", "standard"
    if sources:
        try:
            payload={"language":"Solidity",
                     "sources":{k:{"content":v} for k,v in sources.items()},
                     "settings":{"optimizer":{"enabled":True,"runs":200}}}
            _ = _compile_standard_resilient(payload)
            return True, "Compiled OK", "all_files"
        except Exception as e_all:
            try:
                main_rel = max(sources.items(), key=lambda kv: len(kv[1]))[0]
                _ = compile_source(sources[main_rel])
                return True, f"Compiled OK (single file: {main_rel})", "single_file"
            except Exception as e_single:
                joined = f"SolcError (v{version}, mode=all_files)\n{_err_text(e_all)}\n---\nSolcError (v{version}, mode=single_file)\n{_err_text(e_single)}"
                return False, joined, "all_files"
    return False, "No .sol sources found", "none"

def pick_seed_versions(prev_used: str, sources: Dict[str,str], prev_msg: str) -> List[str]:
    """Build a small, targeted candidate list (seeded by prior message)."""
    lo,hi,legacy = scan_pragmas(sources)
    floor = _feature_floor(sources)
    seeds = []

    # 1) hint from previous error, if present
    hinted = _extract_semver_from_error(prev_msg or "")
    if hinted:
        if vtup(hinted)[1] >= 9: hinted = MAX_SOLC
        if vtup(hinted) < vtup(floor): hinted = floor
        seeds.append(hinted)

    # 2) previous used_version (clamped), if any
    if prev_used:
        v = _clamp_max(prev_used)
        if vtup(v) < vtup(floor): v = floor
        if v not in seeds: seeds.append(v)

    # 3) pragma-best inside range
    if hi != (99,99,99):
        best = vstr((hi[0],hi[1],max(0,hi[2]-1)))
        if vtup(best) < vtup(floor): best = floor
        if best not in seeds: seeds.append(best)

    # 4) legacy unnamed fallback → prefer 0.5.17 if allowed/no pragma
    if legacy:
        if hi == (99,99,99) or ((0,5,17) < hi and (0,5,0) >= lo):
            if "0.5.17" not in seeds: seeds.append("0.5.17")

    # 5) if still empty, use floor, then a modern default
    if not seeds:
        seeds = [floor, "0.8.21"]

    # clamp all to <= MAX_SOLC and dedup
    out = []
    seen = set()
    for s in seeds:
        s2 = _clamp_max(s)
        if s2 not in seen:
            out.append(s2); seen.add(s2)
    return out

def compile_project(project_dir: Path, prev_used: str, prev_msg: str):
    std = _load_standard_input(project_dir)
    if std and isinstance(std.get("sources"), dict) and std["sources"]:
        src_map = {k: (v.get("content") if isinstance(v, dict) else str(v)) for k, v in std["sources"].items()}
        sources_for_all = None
    else:
        src_map = _collect_sources_from_dir(project_dir)
        sources_for_all = src_map

    n_sources = len(src_map)
    seeds = pick_seed_versions(prev_used, src_map, prev_msg)

    # First pass: try seeds
    tried = set()
    last_msg = ""
    for ver in seeds:
        if ver in tried: continue
        tried.add(ver)
        ok, msg, mode = _compile_once(ver, std, sources_for_all)
        if ok:
            tag = "seed"
            return True, f"Compiled OK ({tag} v{ver}, mode={mode})", ver, n_sources, mode
        last_msg = msg

        # if SPDX error, try a quick minor ladder (0.8→0.7→0.6→0.5→0.4)
        if _looks_spdx_error(msg):
            for M2,m2 in MINOR_ORDER:
                vv = vstr(latest_in_minor(M2,m2))
                if vv in tried: continue
                tried.add(vv)
                ok2, msg2, mode2 = _compile_once(vv, std, sources_for_all)
                if ok2:
                    return True, f"Compiled OK (spdx_downgrade v{vv}, mode={mode2})", vv, n_sources, mode2
                last_msg = msg + "\n---\n" + msg2

        # if version mismatch and pragma info exists, jump directly to pragma-best
        lo,hi,_ = scan_pragmas(src_map)
        if _looks_version_mismatch(msg) and hi != (99,99,99):
            pb = vstr((hi[0],hi[1],max(0,hi[2]-1)))
            if pb not in tried:
                tried.add(pb)
                ok3, msg3, mode3 = _compile_once(pb, std, sources_for_all)
                if ok3:
                    return True, f"Compiled OK (pragma_best v{pb}, mode={mode3})", pb, n_sources, mode3
                last_msg = last_msg + "\n---\n" + msg3

    return False, (last_msg or "Compilation failed (no further detail)"), (seeds[0] if seeds else ""), n_sources, "none"

# ---------------- run on selected rows ONLY (in-place update) ----------------
df = pd.read_csv(CSV_PATH)

# Build a mask by original_idx; keep rows order intact
mask = df["original_idx"].isin(RETRY_ORIGINAL_IDX)
subset = df.loc[mask].copy()

print(f"[info] Will retry {len(subset)} rows out of {len(df)} total.")

updated = 0
t_all0 = time.perf_counter()

for i, row in subset.iterrows():
    addr = str(row["contract_address"]).strip().lower()
    proj_dir = Path(row["project_dir"]) if isinstance(row.get("project_dir"), str) else (PROJECTS_DIR / addr)
    prev_used = str(row.get("used_version") or "").strip()
    prev_msg  = str(row.get("recompile_message") or "")

    t0 = time.perf_counter()
    if not proj_dir.exists():
        df.at[i, "recompile_ok"]       = False
        df.at[i, "recompile_message"]  = "Project directory not found"
        df.at[i, "used_version"]       = prev_used
        df.at[i, "n_sources"]          = 0
        df.at[i, "mode"]               = "none"
        df.at[i, "elapsed_s"]          = round(time.perf_counter() - t0, 3)
        continue

    ok, msg, used, nsrc, mode = compile_project(proj_dir, prev_used, prev_msg)

    # UPDATE ONLY EXISTING COLUMNS FOR THIS ROW
    df.at[i, "recompile_ok"]      = bool(ok)
    df.at[i, "recompile_message"] = str(msg)
    df.at[i, "used_version"]      = used
    df.at[i, "n_sources"]         = int(nsrc)
    df.at[i, "mode"]              = mode
    df.at[i, "elapsed_s"]         = round(time.perf_counter() - t0, 3)
    updated += 1

# Save back to the SAME FILE (no new rows/cols)
df.to_csv(CSV_PATH, index=False)
print(f"[OK] Updated {updated} rows in: {CSV_PATH}  (elapsed {round(time.perf_counter()-t_all0,2)}s)")


In [ ]:
# Targeted, aggressive recompile for selected rows ONLY (no Solidity edits, in-place CSV update)
# - Adds a progress bar via tqdm (falls back to periodic prints if tqdm unavailable)
# - Updates only: recompile_ok, recompile_message, used_version, n_sources, mode, elapsed_s

import re, json, time
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import pandas as pd
from solcx import (
    install_solc,
    set_solc_version,
    compile_source,
    compile_standard,
    get_installed_solc_versions,
)
from solcx.exceptions import SolcError

# Progress bar
try:
    from tqdm.auto import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

# ---------------- user config ----------------
CSV_PATH        = "compilation_results_final.csv"
PROJECTS_DIR    = Path("contracts")
MAX_SOLC        = "0.8.30"  # clamp bogus or >0.8 minors

# If the retry list isn't defined yet, define an empty one (fill it yourself)
try:
    RETRY_ORIGINAL_IDX
except NameError:
    RETRY_ORIGINAL_IDX = []  # <-- put your ~100 indices here if not already set

# ---------------- regexes & constants ----------------
SEMVER3   = re.compile(r"(\d+)\.(\d+)\.(\d+)")
PRAGMA    = re.compile(r"pragma\s+solidity\s+([^;]+);", re.IGNORECASE)
OP_VER    = re.compile(r"([~^]|>=|<=|>|<)?\s*(\d+)\.(\d+)\.(\d+)")
ERR_PRAGMA_SEMVER = re.compile(r"pragma\s+solidity[^0-9]*?(\d+\.\d+\.\d+)", re.IGNORECASE)

LEGACY_FALLBACK_RE = re.compile(r"function\s*\(\)\s*(external|public)", re.IGNORECASE)

LATEST_PATCH = {(0,4):26,(0,5):17,(0,6):12,(0,7):6,(0,8):30}
MINOR_LADDER = [(0,8),(0,7),(0,6),(0,5),(0,4)]  # descending preference

# ---------------- tiny semver helpers ----------------
def vtup(s: str):
    m = SEMVER3.search(s or "")
    return tuple(map(int, m.groups())) if m else (0,0,0)

def vstr(t): return f"{t[0]}.{t[1]}.{t[2]}"

def latest_in_minor(M,m): return (M,m, LATEST_PATCH.get((M,m), 0))

def _semver_tuple(v: str):
    m = SEMVER3.search(v or "")
    return tuple(map(int, m.groups())) if m else (0,0,0)

def _clamp_max(v: str, max_v: str = MAX_SOLC) -> str:
    return max_v if _semver_tuple(v) > _semver_tuple(max_v) else v

# ---------------- pragma range parsing ----------------
def parse_pragma_expr(expr: str):
    lo,hi = (0,0,0),(99,99,99)  # half-open [lo, hi)
    toks = OP_VER.findall(expr)
    if not toks: return lo,hi
    vmax = lambda a,b: a if a>b else b
    vmin = lambda a,b: a if a<b else b
    for op,a,b,c in toks:
        vt = (int(a),int(b),int(c))
        if op in ("",None): lo=vmax(lo,vt); hi=vmin(hi,(vt[0],vt[1],vt[2]+1))
        elif op in ("^","~"): lo=vmax(lo,vt); hi=vmin(hi,(vt[0],vt[1]+1,0))
        elif op==">=": lo=vmax(lo,vt)
        elif op==">":  lo=vmax(lo,(vt[0],vt[1],vt[2]+1))
        elif op=="<=": hi=vmin(hi,(vt[0],vt[1],vt[2]+1))
        elif op=="<":  hi=vmin(hi,vt)
    return lo,hi

def scan_pragmas(sources: Dict[str,str]) -> Tuple[Tuple[int,int,int],Tuple[int,int,int],bool]:
    lo,hi = (0,0,0),(99,99,99)
    legacy=False
    for content in sources.values():
        if LEGACY_FALLBACK_RE.search(content or ""):
            legacy=True
        for pm in PRAGMA.finditer(content or ""):
            a,b = parse_pragma_expr(pm.group(1))
            lo = a if a>lo else lo
            hi = b if b<hi else hi
    return lo,hi,legacy

def minors_between(lo: Tuple[int,int,int], hi: Tuple[int,int,int]) -> List[Tuple[int,int]]:
    """Return allowed minors (0,m) with any overlap in [lo,hi)."""
    allowed = []
    lo_minor = lo[1] if lo[0]==0 else 99
    hi_minor = (hi[1]-1) if (hi[0]==0 and hi[1]>0) else (hi[1] if hi[0]==0 else -1)
    if lo_minor<=hi_minor and hi_minor<=8:
        for m in range(hi_minor, lo_minor-1, -1):
            if (0,m) in LATEST_PATCH:
                allowed.append((0,m))
    return allowed

# ---------------- feature detection → floor ----------------
def _feature_floor(sources: Dict[str, str]) -> str:
    """Conservative minimum compiler implied by features."""
    blob = "\n".join(sources.values())
    floor = (0,4,0)
    if re.search(r"\bcalldata\b", blob): floor = max(floor, (0,5,0))
    if re.search(r"\bimmutable\b", blob): floor = max(floor, (0,6,5))
    if re.search(r"\babstract\s+contract\b", blob) or re.search(r"\boverride\b", blob) or \
       re.search(r"\bvirtual\b", blob) or re.search(r"\breceive\s*\(\)\s*external", blob) or \
       re.search(r"\btry\b\s+[A-Za-z_]\w*\s*\(", blob) or re.search(r"\bcatch\b", blob):
        floor = max(floor, (0,6,0))
    if re.search(r"\berror\s+[A-Za-z_]\w*\s*\(", blob): floor = max(floor, (0,8,4))
    return ".".join(map(str, floor))

# ---------------- IO helpers ----------------
def _collect_sources_from_dir(project_dir: Path) -> Dict[str, str]:
    sources: Dict[str, str] = {}
    for p in project_dir.rglob("*.sol"):
        rel = p.relative_to(project_dir).as_posix()
        try: txt = p.read_text()
        except Exception: txt = ""
        sources[rel] = txt
    return sources

def _load_standard_input(project_dir: Path) -> Optional[dict]:
    f = project_dir / "standard_input.json"
    if not f.exists(): return None
    try:
        raw = json.loads(f.read_text())
        return raw if isinstance(raw, dict) else None
    except Exception:
        return None

# ---------------- compilation plumbing ----------------
_current_solc: Optional[str] = None
def _ensure_solc(version: str) -> Optional[str]:
    global _current_solc
    try:
        version = _clamp_max(version)
        if _current_solc == version: return None
        installed = {str(v) for v in get_installed_solc_versions()}
        if version not in installed: install_solc(version)
        set_solc_version(version)
        _current_solc = version
        return None
    except Exception as e:
        return f"Failed to set solc {version}: {e}"

def _err_text(e: Exception) -> str:
    if isinstance(e, SolcError):
        stderr = getattr(e, "stderr_data", None) or getattr(e, "stderr", None) or ""
        stdout = getattr(e, "stdout_data", None) or getattr(e, "stdout", None) or ""
        tail = lambda s: s[-1400:] if isinstance(s,str) else ""
        return f"--- stderr (tail) ---\n{tail(stderr)}\n--- stdout (tail) ---\n{tail(stdout)}"
    return str(e)

def _looks_spdx_error(txt_or_exc) -> bool:
    txt = txt_or_exc if isinstance(txt_or_exc, str) else _err_text(txt_or_exc)
    return "Invalid SPDX license identifier" in txt

def _looks_version_mismatch(txt_or_exc) -> bool:
    txt = txt_or_exc if isinstance(txt_or_exc, str) else _err_text(txt_or_exc)
    return ("requires different compiler version" in txt) or ("Source file requires different compiler version" in txt)

def _extract_semver_from_error(txt_or_exc) -> Optional[str]:
    txt = txt_or_exc if isinstance(txt_or_exc, str) else _err_text(txt_or_exc)
    m = ERR_PRAGMA_SEMVER.search(txt)
    if m:
        v = m.group(1)
        if vtup(v)[1] >= 9:  # clamp bogus 0.9.x suggestions
            v = MAX_SOLC
        return _clamp_max(v)
    return None

def _compile_standard_resilient(payload: dict):
    try:
        return compile_standard(payload)
    except SolcError as e1:
        txt = _err_text(e1)
        if "Invalid EVM version requested" in txt:
            st = dict(payload.get("settings") or {})
            st.pop("evmVersion", None)
            p2 = dict(payload); p2["settings"] = st
            return compile_standard(p2)
        raise

def _compile_once(version: str, std: Optional[dict], sources: Optional[Dict[str,str]], settings_profiles: List[dict]) -> Tuple[bool,str,str]:
    err = _ensure_solc(version)
    if err: return False, err, "init"
    last_msg = ""
    for st in settings_profiles:
        if std and isinstance(std.get("sources"), dict) and std["sources"]:
            try:
                payload = {"language":"Solidity",
                           "sources":{k:({"content": (v.get("content") if isinstance(v, dict) else str(v))})
                                      for k,v in std["sources"].items()},
                           "settings": st}
                _ = _compile_standard_resilient(payload)
                return True, "Compiled OK", "standard"
            except Exception as e:
                last_msg = f"SolcError (v{version}, mode=standard)\n{_err_text(e)}"
        elif sources:
            try:
                payload={"language":"Solidity",
                         "sources":{k:{"content":v} for k,v in sources.items()},
                         "settings": st}
                _ = _compile_standard_resilient(payload)
                return True, "Compiled OK", "all_files"
            except Exception as e_all:
                try:
                    main_rel = max(sources.items(), key=lambda kv: len(kv[1]))[0]
                    _ = compile_source(sources[main_rel])
                    return True, f"Compiled OK (single file: {main_rel})", "single_file"
                except Exception as e_single:
                    last_msg = f"SolcError (v{version}, mode=all_files)\n{_err_text(e_all)}\n---\nSolcError (v{version}, mode=single_file)\n{_err_text(e_single)}"
        else:
            return False, "No .sol sources found", "none"
    return False, last_msg or "Compilation failed (no further detail)", "none"

def settings_profiles_for(v: str) -> List[dict]:
    base = {"optimizer":{"enabled":True,"runs":200}}
    profiles = [base]
    M,m,_ = vtup(v)
    if (M,m) in {(0,8),(0,7)}:
        for evm in ["istanbul","berlin","london","paris"]:
            profiles.append({"optimizer":{"enabled":True,"runs":200},"evmVersion":evm})
    elif (M,m) in {(0,6),(0,5),(0,4)}:
        for evm in ["byzantium","petersburg","istanbul"]:
            profiles.append({"optimizer":{"enabled":True,"runs":200},"evmVersion":evm})
    profiles.append({"optimizer":{"enabled":False}})
    return profiles

# ---------------- version candidate builder (aggressive) ----------------
def build_candidates(prev_used: str, prev_msg: str, src_map: Dict[str,str]) -> List[str]:
    lo,hi,legacy = scan_pragmas(src_map)
    floor = _feature_floor(src_map)
    cands: List[str] = []
    seen = set()

    def add(v):
        v2 = _clamp_max(v)
        if vtup(v2) < vtup(floor): v2 = floor
        if v2 not in seen:
            seen.add(v2); cands.append(v2)

    hinted = _extract_semver_from_error(prev_msg or "")
    if hinted: add(hinted)
    if prev_used: add(prev_used)
    if hi != (99,99,99):
        best = vstr((hi[0],hi[1],max(0,hi[2]-1)))
        add(best)
    add(floor)

    allowed_minors = minors_between(lo,hi) if hi != (99,99,99) else []
    for (M,m) in allowed_minors:
        add(vstr(latest_in_minor(M,m)))
        if m == 8:
            for p in [30,29,28,27,26,25,24,23,22,21,20,19,18,17,16,15]:
                add(f"0.8.{p}")

    for (M,m) in MINOR_LADDER:
        add(vstr(latest_in_minor(M,m)))

    return cands[:60]

# ---------------- main per-project compile ----------------
def compile_project(project_dir: Path, prev_used: str, prev_msg: str):
    std = _load_standard_input(project_dir)
    if std and isinstance(std.get("sources"), dict) and std["sources"]:
        src_map = {k: (v.get("content") if isinstance(v, dict) else str(v)) for k, v in std["sources"].items()}
        sources_for_all = None
    else:
        src_map = _collect_sources_from_dir(project_dir)
        sources_for_all = src_map

    n_sources = len(src_map)
    if n_sources == 0:
        return False, "No .sol sources found", "", 0, "none"

    candidates = build_candidates(prev_used, prev_msg, src_map)
    last_msg = ""
    tried = set()

    for ver in candidates:
        if ver in tried: continue
        tried.add(ver)

        profiles = settings_profiles_for(ver)
        ok, msg, mode = _compile_once(ver, std, sources_for_all, profiles)
        if ok:
            tag = "seed" if ver == candidates[0] else "aggressive"
            return True, f"Compiled OK ({tag} v{ver}, mode={mode})", ver, n_sources, mode

        if _looks_spdx_error(msg):
            for (M2,m2) in MINOR_LADDER[1:]:
                vv = vstr(latest_in_minor(M2,m2))
                if vv in tried: continue
                tried.add(vv)
                profiles2 = settings_profiles_for(vv)
                ok2, msg2, mode2 = _compile_once(vv, std, sources_for_all, profiles2)
                if ok2:
                    return True, f"Compiled OK (spdx_downgrade v{vv}, mode={mode2})", vv, n_sources, mode2
                msg = msg + "\n---\n" + msg2

        lo,hi,_ = scan_pragmas(src_map)
        if _looks_version_mismatch(msg) and hi != (99,99,99):
            pb = vstr((hi[0],hi[1],max(0,hi[2]-1)))
            if pb not in tried:
                tried.add(pb)
                profiles_pb = settings_profiles_for(pb)
                ok3, msg3, mode3 = _compile_once(pb, std, sources_for_all, profiles_pb)
                if ok3:
                    return True, f"Compiled OK (pragma_best v{pb}, mode={mode3})", pb, n_sources, mode3
                msg = msg + "\n---\n" + msg3

        last_msg = msg

    return False, (last_msg or "Compilation failed (no further detail)"), candidates[0] if candidates else "", n_sources, "none"

# ---------------- run on selected rows ONLY (in-place) ----------------
if not RETRY_ORIGINAL_IDX:
    print("[info] RETRY_ORIGINAL_IDX is empty. Nothing to do.")
else:
    df = pd.read_csv(CSV_PATH)
    mask = df["original_idx"].isin(RETRY_ORIGINAL_IDX)
    subset = df.loc[mask].copy()

    total = len(subset)
    print(f"[info] Will retry {total} rows out of {len(df)} total…")

    updated = 0
    t_all0 = time.perf_counter()

    # Choose iterator with progress
    iterator = subset.index.tolist()
    if _HAS_TQDM:
        pbar = tqdm(total=total, desc="Recompiling (targeted)", unit="row")
    else:
        pbar = None

    for k, i in enumerate(iterator, 1):
        row = subset.loc[i]
        addr = str(row["contract_address"]).strip().lower()
        proj_dir = Path(row["project_dir"]) if isinstance(row.get("project_dir"), str) and row["project_dir"] else (PROJECTS_DIR / addr)
        prev_used = str(row.get("used_version") or "").strip()
        prev_msg  = str(row.get("recompile_message") or "").strip()

        t0 = time.perf_counter()
        if not proj_dir.exists():
            df.at[i, "recompile_ok"]      = False
            df.at[i, "recompile_message"] = "Project directory not found"
            df.at[i, "used_version"]      = prev_used
            df.at[i, "n_sources"]         = 0
            df.at[i, "mode"]              = "none"
            df.at[i, "elapsed_s"]         = round(time.perf_counter() - t0, 3)
        else:
            ok, msg, used, nsrc, mode = compile_project(proj_dir, prev_used, prev_msg)
            df.at[i, "recompile_ok"]      = bool(ok)
            df.at[i, "recompile_message"] = str(msg)
            df.at[i, "used_version"]      = used
            df.at[i, "n_sources"]         = int(nsrc)
            df.at[i, "mode"]              = mode
            df.at[i, "elapsed_s"]         = round(time.perf_counter() - t0, 3)
        updated += 1

        if pbar:
            pbar.update(1)
            pbar.set_postfix(addr=addr[-6:], ok=bool(df.at[i, "recompile_ok"]), ver=df.at[i, "used_version"] or "-")
        elif k % 10 == 0 or k == total:
            print(f"[{k}/{total}] last={addr[-6:]} ok={bool(df.at[i,'recompile_ok'])} ver={df.at[i,'used_version'] or '-'}")

    if pbar:
        pbar.close()

    df.to_csv(CSV_PATH, index=False)
    print(f"[OK] Updated {updated} rows in: {CSV_PATH}  (elapsed {round(time.perf_counter()-t_all0,2)}s)")


In [ ]:
# Retry ONLY selected rows in-place, using an entry-file strategy (no source edits).
# - Updates *only* the selected rows in compilation_results_final.csv
# - Prefers standard_input.json; else compiles one entry file with import resolution (compile_files)
# - Smarter version seeding from: prior error, prior used_version, pragma ranges, feature floor
# - Handles: version mismatch hops, SPDX minor downgrades, drops evmVersion when needed
# - Never adds columns / rows

import re, json, time
from pathlib import Path
from typing import Dict, Optional, Tuple, List
import pandas as pd

from solcx import (
    install_solc, set_solc_version,
    compile_standard, compile_source, compile_files,
    get_installed_solc_versions,
)
from solcx.exceptions import SolcError

try:
    from tqdm.auto import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

# ---------- CONFIG ----------
CSV_PATH        = "compilation_results_final.csv"
PROJECTS_DIR    = Path("contracts")
RETRY_ORIGINAL_IDX: List[int] = []   # <-- leave empty to auto-use all failing rows

# ---------- Regex / helpers ----------
SEMVER3 = re.compile(r"(\d+)\.(\d+)\.(\d+)")
PRAGMA  = re.compile(r"pragma\s+solidity\s+([^;]+);", re.IGNORECASE)
OP_VER  = re.compile(r"([~^]|>=|<=|>|<)?\s*(\d+)\.(\d+)\.(\d+)")
ERR_PRAGMA_SEMVER = re.compile(r"pragma\s+solidity[^0-9]*?(\d+\.\d+\.\d+)", re.IGNORECASE)

LATEST_PATCH = {(0,4):26,(0,5):17,(0,6):12,(0,7):6,(0,8):30}
MINOR_ORDER  = [(0,8),(0,7),(0,6),(0,5),(0,4)]
MAX_SOLC     = "0.8.30"

LEGACY_FALLBACK_RE = re.compile(r"function\s*\(\)\s*(external|public)\s*payable", re.IGNORECASE)

def vtup(s: str):
    m = SEMVER3.search(s or "")
    return tuple(map(int, m.groups())) if m else (0,0,0)

def vstr(t): return f"{t[0]}.{t[1]}.{t[2]}"

def latest_in_minor(M,m): return (M,m, LATEST_PATCH.get((M,m), 0))

def parse_pragma_expr(expr: str):
    lo,hi = (0,0,0),(99,99,99)  # half-open [lo,hi)
    toks = OP_VER.findall(expr or "")
    if not toks: return lo,hi
    vmax = lambda a,b: a if a>b else b
    vmin = lambda a,b: a if a<b else b
    for op,a,b,c in toks:
        vt = (int(a),int(b),int(c))
        if op in ("",None): lo=vmax(lo,vt); hi=vmin(hi,(vt[0],vt[1],vt[2]+1))
        elif op=="^" or op=="~": lo=vmax(lo,vt); hi=vmin(hi,(vt[0],vt[1]+1,0))
        elif op==">=": lo=vmax(lo,vt)
        elif op==">":  lo=vmax(lo,(vt[0],vt[1],vt[2]+1))
        elif op=="<=": hi=vmin(hi,(vt[0],vt[1],vt[2]+1))
        elif op=="<":  hi=vmin(hi,vt)
    return lo,hi

def scan_pragmas_in_text(text: str):
    lo,hi = (0,0,0),(99,99,99)
    have=False
    for pm in PRAGMA.finditer(text or ""):
        a,b = parse_pragma_expr(pm.group(1))
        have=True
        lo = a if a>lo else lo
        hi = b if b<hi else hi
    return (lo,hi) if have else ((0,0,0),(99,99,99))

def _collect_sources_from_dir(project_dir: Path) -> Dict[str,str]:
    out={}
    for p in project_dir.rglob("*.sol"):
        try: out[p.relative_to(project_dir).as_posix()] = p.read_text()
        except Exception: out[p.relative_to(project_dir).as_posix()] = ""
    return out

def _load_standard_input(project_dir: Path) -> Optional[dict]:
    f = project_dir / "standard_input.json"
    if not f.exists(): return None
    try:
        raw = json.loads(f.read_text())
        return raw if isinstance(raw, dict) else None
    except Exception:
        return None

def _feature_floor(blob: str) -> str:
    floor = (0,4,0)
    if re.search(r"\bcalldata\b", blob): floor = max(floor, (0,5,0))
    if re.search(r"\bimmutable\b", blob): floor = max(floor, (0,6,5))
    if re.search(r"\babstract\s+contract\b|\boverride\b|\bvirtual\b|\breceive\s*\(\)\s*external|\btry\b\s+\w+\s*\(|\bcatch\b", blob):
        floor = max(floor, (0,6,0))
    if re.search(r"\berror\s+\w+\s*\(", blob): floor = max(floor, (0,8,4))
    return ".".join(map(str, floor))

def _clamp_max(v: str, max_v: str = MAX_SOLC) -> str:
    return max_v if vtup(v) > vtup(max_v) else v

_current_solc: Optional[str] = None
def _ensure_solc(version: str) -> Optional[str]:
    global _current_solc
    try:
        if _current_solc == version: return None
        installed = {str(v) for v in get_installed_solc_versions()}
        if version not in installed: install_solc(version)
        set_solc_version(version)
        _current_solc = version
        return None
    except Exception as e:
        return f"Failed to set solc {version}: {e}"

def _err_text(e: Exception) -> str:
    if isinstance(e, SolcError):
        stderr = getattr(e, "stderr_data", None) or getattr(e, "stderr", None) or ""
        stdout = getattr(e, "stdout_data", None) or getattr(e, "stdout", None) or ""
        tail = lambda s: s[-1200:] if isinstance(s,str) else ""
        return f"--- stderr (tail) ---\n{tail(stderr)}\n--- stdout (tail) ---\n{tail(stdout)}"
    return str(e)

def _looks_spdx_error(msg: str) -> bool:
    return "Invalid SPDX license identifier" in (msg or "")

def _looks_version_mismatch(msg: str) -> bool:
    s = msg or ""
    return ("requires different compiler version" in s) or ("Source file requires different compiler version" in s)

def _extract_semver_from_error(msg: str) -> Optional[str]:
    m = ERR_PRAGMA_SEMVER.search(msg or "")
    if not m: return None
    v = _clamp_max(m.group(1))
    return v

# ---------- Entry file selection (avoid mixing stray sources) ----------
def _file_pragma_max_tuple(text: str) -> Tuple[int,int,int]:
    lo,hi = scan_pragmas_in_text(text)
    if hi == (99,99,99):   # no pragma: treat as very low
        return (-1,-1,-1)
    return (hi[0], hi[1], max(0, hi[2]-1))

def _entry_candidates(project_dir: Path, limit=3) -> List[Path]:
    files = list(project_dir.rglob("*.sol"))
    scored = []
    for p in files:
        try:
            t = p.read_text()
        except Exception:
            t = ""
        pragma_t = _file_pragma_max_tuple(t)
        contracts = len(re.findall(r"\bcontract\b", t))
        score = (pragma_t, contracts, len(t))
        scored.append((score, p))
    scored.sort(reverse=True)
    return [p for _, p in scored[:limit]] if scored else []

# ---------- Seeds & compilation ----------
def _seed_versions(prev_used: str, prev_msg: str, blob: str, entry_pragmas: List[str]) -> List[str]:
    seeds=[]
    hinted = _extract_semver_from_error(prev_msg)
    floor  = _clamp_max(_feature_floor(blob))
    if hinted:
        if vtup(hinted) < vtup(floor): hinted = floor
        seeds.append(hinted)
    if prev_used:
        v = _clamp_max(prev_used)
        if vtup(v) < vtup(floor): v = floor
        if v not in seeds: seeds.append(v)
    # pragma-best from entry files
    for expr in entry_pragmas:
        lo,hi = parse_pragma_expr(expr)
        if hi != (99,99,99):
            best = vstr((hi[0],hi[1],max(0,hi[2]-1)))
            if vtup(best) < vtup(floor): best = floor
            if best not in seeds: seeds.append(best)
    # generic floor + modern default
    if floor not in seeds: seeds.append(floor)
    if "0.8.21" not in seeds: seeds.append("0.8.21")
    # minor ladder for SPDX fallback
    for M,m in MINOR_ORDER:
        vv = vstr(latest_in_minor(M,m))
        if vv not in seeds: seeds.append(vv)
    # dedup while preserving order
    out, seen = [], set()
    for s in seeds:
        s2=_clamp_max(s)
        if s2 not in seen:
            out.append(s2); seen.add(s2)
    return out

def _compile_standard_resilient(payload: dict):
    try:
        return compile_standard(payload)
    except SolcError as e1:
        txt = _err_text(e1)
        if "Invalid EVM version requested" in txt:
            st = dict(payload.get("settings") or {})
            st.pop("evmVersion", None)
            p2 = dict(payload); p2["settings"]=st
            return compile_standard(p2)
        raise

def _compile_entry_strategy(project_dir: Path, seeds: List[str]) -> Tuple[bool,str,str,str]:
    """
    Try up to 3 entry files with compile_files, across seed versions.
    Returns (ok, message, used_version, mode)
    """
    entries = _entry_candidates(project_dir, limit=3)
    if not entries:
        return False, "No .sol sources found", "", "none"

    # collect pragmas from the top candidates for seeding
    entry_pragmas=[]
    for p in entries:
        try:
            for pm in PRAGMA.finditer(p.read_text() or ""):
                entry_pragmas.append(pm.group(1))
        except Exception:
            pass

    # recompute seeds with entry-specific pragmas
    blob = ""
    for p in project_dir.rglob("*.sol"):
        try: blob += p.read_text() + "\n"
        except Exception: pass
    seeds = _seed_versions("", "", blob, entry_pragmas) if not seeds else seeds

    tried=set()
    last_msg=""
    for ver in seeds:
        if ver in tried: continue
        tried.add(ver)
        err = _ensure_solc(ver)
        if err:
            last_msg = err
            continue
        for entry in entries:
            try:
                # compile only this entry; let solc read imports from disk
                _ = compile_files([str(entry)], base_path=str(project_dir), allow_paths=str(project_dir), optimize=True, optimize_runs=200)
                return True, f"Compiled OK (entry:{entry.name})", ver, "entry_files"
            except Exception as e:
                last_msg = f"SolcError (v{ver}, entry={entry.name})\n{_err_text(e)}"

        # If SPDX error seen, do a quick minor ladder immediately
        if _looks_spdx_error(last_msg):
            for M2,m2 in MINOR_ORDER:
                vv = vstr(latest_in_minor(M2,m2))
                if vv in tried: continue
                tried.add(vv)
                err = _ensure_solc(vv)
                if err: 
                    last_msg = err
                    continue
                for entry in entries:
                    try:
                        _ = compile_files([str(entry)], base_path=str(project_dir), allow_paths=str(project_dir), optimize=True, optimize_runs=200)
                        return True, f"Compiled OK (spdx_downgrade {vv}, entry:{entry.name})", vv, "entry_files"
                    except Exception as e2:
                        last_msg = f"SolcError (v{vv}, entry={entry.name})\n{_err_text(e2)}"

    return False, (last_msg or "Compilation failed (no further detail)"), (seeds[0] if seeds else ""), "none"

def compile_project(project_dir: Path, prev_used: str, prev_msg: str) -> Tuple[bool,str,str,int,str]:
    """
    Try standard_input.json first; else entry-file strategy; else final single-file fallback.
    """
    std = _load_standard_input(project_dir)
    if std and isinstance(std.get("sources"), dict) and std["sources"]:
        # standard path
        seeds = []
        blob = ""
        for k,v in std["sources"].items():
            txt = v.get("content") if isinstance(v, dict) else str(v)
            blob += (txt or "") + "\n"
        seeds = _seed_versions(prev_used, prev_msg, blob, [])
        for ver in seeds:
            err = _ensure_solc(ver)
            if err: last = err; continue
            try:
                settings = dict(std.get("settings") or {})
                if "optimizer" not in settings:
                    settings["optimizer"] = {"enabled": True, "runs": 200}
                payload = {"language":"Solidity",
                           "sources":{k:{"content": (v.get("content") if isinstance(v, dict) else str(v))}
                                      for k,v in std["sources"].items()},
                           "settings":settings}
                _ = _compile_standard_resilient(payload)
                return True, f"Compiled OK (standard)", ver, len(std["sources"]), "standard"
            except Exception as e:
                last = f"SolcError (v{ver}, standard)\n{_err_text(e)}"
        return False, last, seeds[0] if seeds else "", len(std["sources"]), "standard"

    # entry-file strategy
    ok, msg, ver, mode = _compile_entry_strategy(project_dir, seeds=[])
    if ok:
        nsrc = sum(1 for _ in project_dir.rglob("*.sol"))
        return True, msg, ver, nsrc, mode

    # last resort: single-file compile of largest file (with no imports)
    sources = _collect_sources_from_dir(project_dir)
    if sources:
        main_rel = max(sources.items(), key=lambda kv: len(kv[1]))[0]
        # try a small ladder
        blob = "\n".join(sources.values())
        seeds = _seed_versions(prev_used, msg, blob, [])
        for ver in seeds[:8]:
            err = _ensure_solc(ver)
            if err: last = err; continue
            try:
                _ = compile_source(sources[main_rel])
                return True, f"Compiled OK (single file: {main_rel})", ver, len(sources), "single_file"
            except Exception as e:
                last = f"SolcError (v{ver}, single_file={main_rel})\n{_err_text(e)}"
        return False, last, seeds[0] if seeds else "", len(sources), "single_file"

    return False, "No .sol sources found", "", 0, "none"

# ---------- RUN (only selected rows) ----------
df = pd.read_csv(CSV_PATH)

if not RETRY_ORIGINAL_IDX:
    # auto-pick all current failures
    mask_fail = df.get("recompile_ok").astype(str).str.lower().isin(["false","0","no","nan"])
    RETRY_ORIGINAL_IDX = df.loc[mask_fail, "original_idx"].dropna().astype(int).tolist()

mask = df["original_idx"].isin(RETRY_ORIGINAL_IDX)
subset = df.loc[mask].copy()
print(f"[info] Retrying {len(subset)} rows out of {len(df)} total.")

it = subset.iterrows()
if _HAS_TQDM: it = tqdm(it, total=len(subset), desc="Retry (entry-file strategy)", unit="row")

updated=0
for i, row in it:
    addr = str(row["contract_address"]).strip().lower()
    proj_dir = Path(row.get("project_dir") if isinstance(row.get("project_dir"), str) else (PROJECTS_DIR / addr))
    prev_used = str(row.get("used_version") or "").strip()
    prev_msg  = str(row.get("recompile_message") or "")

    t0 = time.perf_counter()
    if not proj_dir.exists():
        df.at[i, "recompile_ok"]      = False
        df.at[i, "recompile_message"] = "Project directory not found"
        df.at[i, "used_version"]      = prev_used
        df.at[i, "n_sources"]         = 0
        df.at[i, "mode"]              = "none"
        df.at[i, "elapsed_s"]         = round(time.perf_counter() - t0, 3)
        continue

    ok, msg, used, nsrc, mode = compile_project(proj_dir, prev_used, prev_msg)

    # Update ONLY existing columns, in-place
    df.at[i, "recompile_ok"]      = bool(ok)
    df.at[i, "recompile_message"] = str(msg)
    df.at[i, "used_version"]      = used
    df.at[i, "n_sources"]         = int(nsrc)
    df.at[i, "mode"]              = mode
    df.at[i, "elapsed_s"]         = round(time.perf_counter() - t0, 3)
    updated += 1

df.to_csv(CSV_PATH, index=False)
print(f"[OK] Updated {updated} rows → {CSV_PATH}")


## Adding new ones instead..

In [ ]:
# ======================================================================
# Replace failing rows with fresh, verified & compiled contracts
#   from ASSERT-KTH/FLAMES_results (100k) and ASSERT-KTH/DISL (decomposed),
#   fetching sources from Etherscan V2.
#
# REQS: pip install pandas py-solc-x datasets huggingface_hub requests tqdm python-dotenv
# ENV in .env:
#   ETHERSCAN_API_KEY=...
#   HUGGINGFACE_TOKEN=...    (or HF_TOKEN / HUGGINGFACEHUB_API_TOKEN)
# ======================================================================

import os, re, json, random, time
from pathlib import Path
from typing import Dict, Optional, Tuple, List, Any

import pandas as pd
import requests
from datasets import load_dataset
from tqdm.auto import tqdm

from solcx import (
    install_solc, set_solc_version, get_installed_solc_versions,
    compile_standard, compile_source
)
from solcx.exceptions import SolcError
from dotenv import load_dotenv
from huggingface_hub import HfFolder

# ---------------- CONFIG ----------------
IN_CSV       = "compilation_results_final_complete.csv"
OUT_CSV      = IN_CSV
PROJECTS_DIR = Path("contracts")

HF_FLAMES       = "ASSERT-KTH/FLAMES_results"
HF_FLAMES_CFG   = "100k"
HF_DISL         = "ASSERT-KTH/DISL"
HF_DISL_CFG     = "decomposed"            # row position == original_idx for this config

MAX_SOLC        = "0.8.30"
RNG_SEED        = 42
DEBUG_FULL_ETHERSCAN_JSON = True          # print full JSON for failures
V2_BASE         = "https://api.etherscan.io/v2/api"
CHAIN_ID        = 1                       # Ethereum mainnet

load_dotenv()

def _require_env(var_names: List[str], label: str) -> str:
    for v in var_names:
        val = os.environ.get(v, "").strip()
        if val:
            return val
    raise SystemExit(f"Please set {label} in your .env (one of {', '.join(var_names)}).")

ETHERSCAN_API_KEY = _require_env(["ETHERSCAN_API_KEY"], "ETHERSCAN_API_KEY")
HF_TOKEN = _require_env(
    ["HUGGINGFACE_TOKEN", "HF_TOKEN", "HUGGINGFACEHUB_API_TOKEN"],
    "a Hugging Face access token"
)

# Register token for both datasets & hub
HfFolder.save_token(HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

# ---------------- small helpers ----------------
SEMVER3 = re.compile(r"(\d+)\.(\d+)\.(\d+)")
def vtup(v: str):
    m = SEMVER3.search(v or "")
    return tuple(map(int, m.groups())) if m else (0,0,0)
def _clamp_max(v: str, max_v: str = MAX_SOLC) -> str:
    return max_v if vtup(v) > vtup(max_v) else v

def ensure_solc(version: str) -> None:
    version = _clamp_max(version)
    installed = {str(v) for v in get_installed_solc_versions()}
    if version not in installed:
        install_solc(version)
    set_solc_version(version)

def err_text(e: Exception) -> str:
    if isinstance(e, SolcError):
        stderr = getattr(e, "stderr_data", None) or getattr(e, "stderr", None) or ""
        stdout = getattr(e, "stdout_data", None) or getattr(e, "stdout", None) or ""
        tail = lambda s: s[-1200:] if isinstance(s,str) else ""
        return f"--- stderr (tail) ---\n{tail(stderr)}\n--- stdout (tail) ---\n{tail(stdout)}"
    return str(e)

# ---------------- Etherscan V2 fetch & parse ----------------
def _etherscan_v2_get(path_params: dict, row_no: int, attempt: int, address: str) -> Optional[dict]:
    params = {
        "chainid": CHAIN_ID,
        "apikey": ETHERSCAN_API_KEY,
        **path_params,
    }
    t0 = time.perf_counter()
    r = requests.get(V2_BASE, params=params, timeout=30)
    elapsed = time.perf_counter() - t0
    try:
        data = r.json()
    except Exception:
        data = {"status": "0", "message": f"HTTP {r.status_code}", "result": r.text[:2000]}
    print(f"[row#{row_no}] Etherscan V2 GET address={address} (attempt {attempt}) elapsed={elapsed:.3f}s")
    if DEBUG_FULL_ETHERSCAN_JSON:
        try:
            print(json.dumps(data, indent=2)[:4000])
        except Exception:
            print(str(data)[:4000])
    return data

def etherscan_get_source_v2(address: str, row_no: int) -> Optional[dict]:
    """
    V2: /v2/api?chainid=1&module=contract&action=getsourcecode&address=...
    - Follows proxy automatically if Proxy == "1"
    - Gentle backoff on 'Max rate limit reached' or transient NOTOKs
    """
    address = address.lower()
    MAX_TRIES = 6
    backoff = 1.2
    attempt = 0
    while attempt < MAX_TRIES:
        attempt += 1
        data = _etherscan_v2_get(
            {"module": "contract", "action": "getsourcecode", "address": address},
            row_no=row_no, attempt=attempt, address=address
        )
        status = str(data.get("status"))
        message = str(data.get("message"))
        result = data.get("result")

        # Successful payload has status "1" and a list in "result"
        if status == "1" and isinstance(result, list) and result:
            item = result[0]
            # If verified proxy, follow implementation
            if str(item.get("Proxy", "")).strip() == "1":
                impl = (item.get("Implementation") or "").strip()
                if impl:
                    print(f"[row#{row_no}] proxy detected → following Implementation={impl}")
                    address = impl
                    # loop continues and re-queries implementation
                    continue
            # Verified with non-empty SourceCode?
            if (item.get("SourceCode") or "").strip():
                return item
            else:
                print(f"[row#{row_no}] verified but empty SourceCode; treating as unverified")
                return None

        # Handle rate limiting and transient NOTOKs
        msg = (message or "").lower()
        if "max rate limit" in msg or "please use a valid api" in msg or status == "0":
            sleep_s = min(6.0, backoff)
            print(f"[row#{row_no}] etherscan: NOTOK ({message}); sleeping {sleep_s:.2f}s then retry…")
            time.sleep(sleep_s)
            backoff *= 1.6
            continue

        # Anything else: treat as failure
        return None

    print(f"[row#{row_no}] etherscan: retries exhausted")
    return None

def parse_etherscan_sources(item: dict) -> Tuple[Dict[str,str], Optional[dict], str]:
    comp_version_raw = (item.get("CompilerVersion") or "").strip()
    m = SEMVER3.search(comp_version_raw)
    comp_version = ".".join(m.groups()) if m else ""

    src = (item.get("SourceCode") or "").strip()
    sources_map, settings = {}, None

    def try_json_extract(s):
        s2 = s
        if s2.startswith("{{") and s2.endswith("}}"):
            s2 = s2[1:-1]
        if (s2.startswith('"') and s2.endswith('"')) or (s2.startswith("'") and s2.endswith("'")):
            try:
                s2 = json.loads(s2)
            except Exception:
                pass
        try:
            return json.loads(s2)
        except Exception:
            return None

    obj = try_json_extract(src)
    if obj and isinstance(obj, dict) and isinstance(obj.get("sources"), dict):
        for k, v in obj["sources"].items():
            if isinstance(v, dict) and "content" in v:
                sources_map[k] = v["content"]
            else:
                sources_map[k] = str(v)
        settings = obj.get("settings") if isinstance(obj.get("settings"), dict) else None

    if not sources_map:
        cname = (item.get("ContractName") or "").strip() or "Contract"
        sources_map[f"{cname}.sol"] = src

    return sources_map, settings, comp_version

# ---------------- write sources to disk ----------------
def write_sources(address: str, sources_map: Dict[str,str], settings: Optional[dict]) -> Path:
    proj_dir = PROJECTS_DIR / address.lower()
    proj_dir.mkdir(parents=True, exist_ok=True)
    for rel, content in sources_map.items():
        rel_path = Path(rel)
        safe_rel = Path(*[p for p in rel_path.parts if p not in ("", "/", "..")])
        out_file = proj_dir / safe_rel
        out_file.parent.mkdir(parents=True, exist_ok=True)
        out_file.write_text(content)
    if settings and isinstance(settings, dict):
        std = {"language": "Solidity",
               "sources": {k: {"content": v} for k, v in sources_map.items()},
               "settings": settings}
        (proj_dir / "standard_input.json").write_text(json.dumps(std, indent=2))
    return proj_dir

# ---------------- simple compile (no source edits) ----------------
def compile_written_project(proj_dir: Path, prefer_version: str) -> Tuple[bool,str,str,int,str]:
    stdf = proj_dir / "standard_input.json"
    versions = []
    if prefer_version:
        versions.append(_clamp_max(prefer_version))
    versions += ["0.8.21", "0.8.17", "0.7.6", "0.6.12", "0.5.17", "0.4.26"]

    sources_map = {}
    for p in proj_dir.rglob("*.sol"):
        try: sources_map[p.relative_to(proj_dir).as_posix()] = p.read_text()
        except Exception: sources_map[p.relative_to(proj_dir).as_posix()] = ""
    n_sources = len(sources_map)

    last_msg = ""
    for ver in versions:
        try:
            ensure_solc(ver)
        except Exception as e:
            last_msg = f"Failed to set solc {ver}: {e}"
            continue

        if stdf.exists():
            try:
                std = json.loads(stdf.read_text())
                st = dict(std.get("settings") or {})
                if "optimizer" not in st:
                    st["optimizer"] = {"enabled": True, "runs": 200}
                payload = {
                    "language": "Solidity",
                    "sources": std.get("sources") or {k: {"content": v} for k, v in sources_map.items()},
                    "settings": st,
                }
                _ = compile_standard(payload)
                return True, f"Compiled OK (standard)", ver, n_sources, "standard"
            except Exception as e:
                last_msg = f"SolcError (v{ver}, standard)\n{err_text(e)}"

        if n_sources:
            try:
                payload = {"language": "Solidity",
                           "sources": {k: {"content": v} for k, v in sources_map.items()},
                           "settings": {"optimizer": {"enabled": True, "runs": 200}}}
                _ = compile_standard(payload)
                return True, f"Compiled OK (all_files)", ver, n_sources, "all_files"
            except Exception as e_all:
                try:
                    main_rel = max(sources_map.items(), key=lambda kv: len(kv[1]))[0]
                    _ = compile_source(sources_map[main_rel])
                    return True, f"Compiled OK (single file: {main_rel})", ver, n_sources, "single_file"
                except Exception as e_single:
                    last_msg = f"SolcError (v{ver}, all_files)\n{err_text(e_all)}\n---\nSolcError (v{ver}, single_file)\n{err_text(e_single)}"
        else:
            last_msg = "No .sol sources found"

    return False, (last_msg or "Compilation failed"), (versions[0] if versions else ""), n_sources, "none"

# ---------------- HuggingFace loaders WITH AUTH ----------------
def _load_with_token(path: str, name: Optional[str] = None, split: str = "train"):
    try:
        return load_dataset(path, name, split=split, token=HF_TOKEN) if name \
            else load_dataset(path, split=split, token=HF_TOKEN)
    except TypeError:
        return load_dataset(path, name, split=split, use_auth_token=HF_TOKEN) if name \
            else load_dataset(path, split=split, use_auth_token=HF_TOKEN)

def load_flames():
    ds = _load_with_token(HF_FLAMES, HF_FLAMES_CFG, split="train")
    print(f"[info] Loaded FLAMES ({HF_FLAMES_CFG}) with {len(ds)} rows.")
    return ds

def load_disl():
    ds = _load_with_token(HF_DISL, HF_DISL_CFG, split="train")
    print(f"[info] Loaded DISL config='{HF_DISL_CFG}' with {len(ds)} rows.")
    return ds

# ---------------- main replacement loop ----------------
def main():
    random.seed(RNG_SEED)

    df = pd.read_csv(IN_CSV)
    mask_fail = df.get("recompile_ok").astype(str).str.lower() != "true"
    fail_idxs = df.index[mask_fail].tolist()
    need = len(fail_idxs)
    if need == 0:
        print("[info] No failing rows found. Nothing to replace.")
        return

    print(f"[info] Need to replace {need} failing rows.")

    # avoid duplicates: original_idx already present in our CSV
    have_original = set(pd.to_numeric(df["original_idx"], errors="coerce").dropna().astype(int).tolist())

    flames = load_flames()
    disl = load_disl()

    # Build: POSITION (enumeration idx) -> contract_address
    # (in DISL/decomposed, row position IS the original_idx)
    disl_map: Dict[int, str] = {}
    for pos, row in tqdm(enumerate(disl), total=len(disl), desc="Indexing DISL (by position)"):
        addr = (row.get("contract_address") or "").strip().lower()
        if addr:
            if not addr.startswith("0x"):
                addr = "0x" + addr
            disl_map[pos] = addr

    # Candidates: len(predicate)>70 and oi not in our CSV and must exist in disl_map
    candidates: List[int] = []
    for row in tqdm(flames, desc="Filtering FLAMES"):
        oi = int(row["original_idx"])
        pred = str(row.get("predicate") or "")
        if len(pred) > 70 and oi not in have_original and oi in disl_map:
            candidates.append(oi)

    random.shuffle(candidates)
    print(f"[info] Candidate pool size: {len(candidates)}")

    replaced = 0
    # Pair each failing row with candidates; keep trying next candidate if fetch fails
    for idx_in_list, fail_i in enumerate(fail_idxs, start=1):
        if not candidates:
            print("[warn] Out of candidates.")
            break

        # keep drawing until we find a verified source
        picked = None
        item = None
        tries = 0
        while candidates and tries < 25 and item is None:
            tries += 1
            oi = candidates.pop()        # consume a candidate
            address = disl_map.get(oi)
            print(f"[row#{idx_in_list}/{need}] try#{tries} oi={oi} → addr={address}")
            item = etherscan_get_source_v2(address, row_no=idx_in_list)
            if item is None:
                print(f"[row#{idx_in_list}]   etherscan: unverified or error; continuing…")
                continue
            picked = (oi, address)

        if not picked:
            print(f"[row#{idx_in_list}]   no verified candidate found after {tries} tries; skipping row.")
            continue

        oi, address = picked
        # Parse, write, compile
        try:
            sources_map, settings, prefer_ver = parse_etherscan_sources(item)
            proj_dir = write_sources(address, sources_map, settings)
        except Exception as e:
            print(f"[row#{idx_in_list}] write failed: {e}")
            continue

        ok, msg, used, nsrc, mode = compile_written_project(proj_dir, prefer_ver)

        # Update failing row in place
        df.at[fail_i, "contract_address"]  = address
        df.at[fail_i, "project_dir"]       = str(proj_dir)
        df.at[fail_i, "recompile_ok"]      = bool(ok)
        df.at[fail_i, "recompile_message"] = str(msg)
        df.at[fail_i, "used_version"]      = used
        df.at[fail_i, "n_sources"]         = int(nsrc)
        df.at[fail_i, "mode"]              = mode
        df.at[fail_i, "elapsed_s"]         = 0.0
        df.at[fail_i, "original_idx"]      = int(oi)

        have_original.add(oi)
        replaced += 1
        print(f"[row#{idx_in_list}] ✅ updated -> ok={ok} used={used or '-'} addr={address}")

        # be polite to Etherscan
        time.sleep(0.25)

        if replaced >= need:
            break

    df.to_csv(OUT_CSV, index=False)
    print(f"[DONE] Replaced {replaced} row(s). CSV saved: {OUT_CSV}")

if __name__ == "__main__":
    main()


# Compilation Pipeline Full

In [ ]:
# ============================================================
# One-shot Solidity recompilation pipeline (no source edits)
# ============================================================
# Stages (all enabled by default):
#   1) Smart all-project pass (standard_input -> all_files -> single_file)
#   2) Vendor/remappings-aware retry for failures (compile_files)
#   3) Aggressive retry for remaining failures (version+settings ladder)
#   4) Entry-file strategy for last failures
#
# CSV discipline:
#   - For pass #1 we write OUT_CSV fresh with the standard set of columns.
#   - For passes #2-#4 we UPDATE rows in-place (same columns; no new rows/cols).
#
# Never edits Solidity sources.
#
# Requirements:
#   pip install pandas py-solc-x tqdm
#
# Config below.
# ============================================================

import os, re, json, time
from pathlib import Path
from typing import Dict, Optional, Tuple, List, Iterable

import pandas as pd
from solcx import (
    install_solc, set_solc_version, get_installed_solc_versions,
    compile_standard, compile_source, compile_files
)
from solcx.exceptions import SolcError

# Progress bar
try:
    from tqdm.auto import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

# ---------------- CONFIG ----------------
INPUT_CSV   = "input.csv"                 # input dataset (original)
PROJECTS_DIR= Path("contracts")                 # base dir containing per-address folders
OUT_CSV     = "compilation_results_flames-100k.csv"   # results CSV

RUN_STAGE_1 = True   # smart, no-touch pass over ALL rows (writes OUT_CSV fresh)
RUN_STAGE_2 = True   # vendor/remappings-aware retry on failures (in-place update)
RUN_STAGE_3 = True   # aggressive retry on remaining failures (in-place update)
RUN_STAGE_4 = True   # entry-file strategy on remaining failures (in-place update)

MAX_SOLC    = "0.8.30"  # clamp bogus >0.8.x “hints”
SLEEP_BETWEEN = 0.0

# ---------------- Regex / constants ----------------
SEMVER3 = re.compile(r"(\d+)\.(\d+)\.(\d+)")
PRAGMA  = re.compile(r"pragma\s+solidity\s+([^;]+);", re.IGNORECASE)
OP_VER  = re.compile(r"([~^]|>=|<=|>|<)?\s*(\d+)\.(\d+)\.(\d+)")
ERR_PRAGMA_SEMVER = re.compile(r"pragma\s+solidity[^0-9]*?(\d+\.\d+\.\d+)", re.IGNORECASE)

LATEST_PATCH = {(0,4):26,(0,5):17,(0,6):12,(0,7):6,(0,8):30}
MINOR_ORDER  = [(0,8),(0,7),(0,6),(0,5),(0,4)]
VALID_MINORS = {4,5,6,7,8}
LEGACY_FALLBACK_RE = re.compile(r"function\s*\(\)\s*(external|public)", re.IGNORECASE)

# ---------------- tiny semver helpers ----------------
def vtup(s: str):
    m = SEMVER3.search(s or "")
    return tuple(map(int, m.groups())) if m else (0,0,0)

def vstr(t): return f"{t[0]}.{t[1]}.{t[2]}"

def latest_in_minor(M,m): return (M,m, LATEST_PATCH.get((M,m), 0))

def _clamp_max(v: str, max_v: str = MAX_SOLC) -> str:
    return max_v if vtup(v) > vtup(max_v) else v

# ---------------- pragma range parsing ----------------
def parse_pragma_expr(expr: str):
    lo,hi = (0,0,0),(99,99,99)  # half-open [lo,hi)
    toks = OP_VER.findall(expr or "")
    if not toks: return lo,hi
    vmax = lambda a,b: a if a>b else b
    vmin = lambda a,b: a if a<b else b
    for op,a,b,c in toks:
        vt = (int(a),int(b),int(c))
        if op in ("",None): lo=vmax(lo,vt); hi=vmin(hi,(vt[0],vt[1],vt[2]+1))
        elif op in ("^","~"): lo=vmax(lo,vt); hi=vmin(hi,(vt[0],vt[1]+1,0))
        elif op==">=": lo=vmax(lo,vt)
        elif op==">":  lo=vmax(lo,(vt[0],vt[1],vt[2]+1))
        elif op=="<=": hi=vmin(hi,(vt[0],vt[1],vt[2]+1))
        elif op=="<":  hi=vmin(hi,vt)
    return lo,hi

def scan_pragmas(sources: Dict[str,str]) -> Tuple[Tuple[int,int,int],Tuple[int,int,int],bool]:
    lo,hi = (0,0,0),(99,99,99)
    legacy=False
    for content in sources.values():
        if LEGACY_FALLBACK_RE.search(content or ""):
            legacy=True
        for pm in PRAGMA.finditer(content or ""):
            a,b = parse_pragma_expr(pm.group(1))
            lo = a if a>lo else lo
            hi = b if b<hi else hi
    return lo,hi,legacy

# ---------------- feature detection → floor ----------------
def _feature_floor(sources: Dict[str, str]) -> str:
    blob = "\n".join(sources.values())
    floor = (0,4,0)
    if re.search(r"\bcalldata\b", blob): floor = max(floor, (0,5,0))
    if re.search(r"\bimmutable\b", blob): floor = max(floor, (0,6,5))
    if re.search(r"\babstract\s+contract\b|\boverride\b|\bvirtual\b|\breceive\s*\(\)\s*external|\btry\b\s+\w+\s*\(|\bcatch\b", blob):
        floor = max(floor, (0,6,0))
    if re.search(r"\berror\s+\w+\s*\(", blob): floor = max(floor, (0,8,4))
    return ".".join(map(str, floor))

# ---------------- IO helpers ----------------
def _collect_sources_from_dir(project_dir: Path) -> Dict[str, str]:
    sources: Dict[str, str] = {}
    for p in project_dir.rglob("*.sol"):
        rel = p.relative_to(project_dir).as_posix()
        try: txt = p.read_text()
        except Exception: txt = ""
        sources[rel] = txt
    return sources

def _load_standard_input(project_dir: Path) -> Optional[dict]:
    f = project_dir / "standard_input.json"
    if not f.exists(): return None
    try:
        raw = json.loads(f.read_text())
        return raw if isinstance(raw, dict) else None
    except Exception:
        return None

# ---------------- error helpers ----------------
def _err_text(e: Exception) -> str:
    if isinstance(e, SolcError):
        stderr = getattr(e, "stderr_data", None) or getattr(e, "stderr", None) or ""
        stdout = getattr(e, "stdout_data", None) or getattr(e, "stdout", None) or ""
        tail = lambda s: s[-1400:] if isinstance(s,str) else ""
        return f"--- stderr (tail) ---\n{tail(stderr)}\n--- stdout (tail) ---\n{tail(stdout)}"
    return str(e)

def _looks_spdx_error(txt_or_exc) -> bool:
    txt = txt_or_exc if isinstance(txt_or_exc, str) else _err_text(txt_or_exc)
    return "Invalid SPDX license identifier" in txt

def _looks_version_mismatch(txt_or_exc) -> bool:
    txt = txt_or_exc if isinstance(txt_or_exc, str) else _err_text(txt_or_exc)
    return ("requires different compiler version" in txt) or ("Source file requires different compiler version" in txt)

def _looks_contract_vs_address0(txt_or_exc) -> bool:
    txt = txt_or_exc if isinstance(txt_or_exc, str) else _err_text(txt_or_exc)
    return "Operator !=" in txt and "contract" in txt and "address" in txt

def _looks_override_list_error(txt_or_exc) -> bool:
    txt = txt_or_exc if isinstance(txt_or_exc, str) else _err_text(txt_or_exc)
    return "Function needs to specify overridden contracts" in txt

def _looks_data_location_mismatch(txt_or_exc) -> bool:
    txt = txt_or_exc if isinstance(txt_or_exc, str) else _err_text(txt_or_exc)
    return "Data locations of return variables have to be the same" in txt

def _extract_semver_from_error(txt_or_exc) -> Optional[str]:
    txt = txt_or_exc if isinstance(txt_or_exc, str) else _err_text(txt_or_exc)
    m = ERR_PRAGMA_SEMVER.search(txt)
    if m:
        v = m.group(1)
        if vtup(v)[1] >= 9:  # clamp bogus 0.9.x suggestions
            v = MAX_SOLC
        return _clamp_max(v)
    return None

# ---------------- solc selection ----------------
_current_solc: Optional[str] = None
def _ensure_solc(version: str) -> Optional[str]:
    global _current_solc
    version = _clamp_max(version)
    try:
        if _current_solc == version: return None
        installed = {str(v) for v in get_installed_solc_versions()}
        if version not in installed: install_solc(version)
        set_solc_version(version)
        _current_solc = version
        return None
    except Exception as e:
        return f"Failed to set solc {version}: {e}"

# ---------------- base compile (standard/all/single) ----------------
def _compile_standard_resilient(payload: dict):
    try:
        return compile_standard(payload)
    except SolcError as e1:
        txt = _err_text(e1)
        if "Invalid EVM version requested" in txt:
            st = dict(payload.get("settings") or {})
            st.pop("evmVersion", None)
            p2 = dict(payload); p2["settings"]=st
            return compile_standard(p2)
        raise

def _compile_once(version: str, std: Optional[dict], sources: Optional[Dict[str,str]]) -> Tuple[bool,str,str]:
    err = _ensure_solc(version)
    if err: return False, err, "init"
    if std and isinstance(std.get("sources"), dict) and std["sources"]:
        try:
            settings = dict(std.get("settings") or {})
            if "optimizer" not in settings:
                settings["optimizer"] = {"enabled": True, "runs": 200}
            payload = {"language":"Solidity",
                       "sources":{k:({"content": (v.get("content") if isinstance(v, dict) else str(v))})
                                  for k,v in std["sources"].items()},
                       "settings":settings}
            _ = _compile_standard_resilient(payload)
            return True, "Compiled OK", "standard"
        except Exception as e:
            return False, f"SolcError (v{version}, mode=standard)\n{_err_text(e)}", "standard"
    if sources:
        try:
            payload={"language":"Solidity",
                     "sources":{k:{"content":v} for k,v in sources.items()},
                     "settings":{"optimizer":{"enabled":True,"runs":200}}}
            _ = _compile_standard_resilient(payload)
            return True, "Compiled OK", "all_files"
        except Exception as e_all:
            try:
                main_rel = max(sources.items(), key=lambda kv: len(kv[1]))[0]
                _ = compile_source(sources[main_rel])
                return True, f"Compiled OK (single file: {main_rel})", "single_file"
            except Exception as e_single:
                joined = f"SolcError (v{version}, mode=all_files)\n{_err_text(e_all)}\n---\nSolcError (v{version}, mode=single_file)\n{_err_text(e_single)}"
                return False, joined, "all_files"
    return False, "No .sol sources found", "none"

def is_reasonable_csv_version(v: str) -> bool:
    m = SEMVER3.search(v or "")
    if not m: return False
    M,mn,p = map(int, m.groups())
    if M!=0 or mn not in VALID_MINORS: return False
    return 0 <= p <= LATEST_PATCH.get((M,mn), -1)

def pick_initial_version(csv_version: str, sources: Dict[str,str]) -> str:
    if isinstance(csv_version,str) and csv_version.strip() and is_reasonable_csv_version(csv_version):
        return ".".join(SEMVER3.search(csv_version).groups())
    lo,hi,legacy = scan_pragmas(sources)
    if hi != (99,99,99):  # have pragma info → choose top of range
        cand = (hi[0],hi[1],max(0,hi[2]-1))
        if cand < lo:
            cand = latest_in_minor(lo[0], lo[1])
        return vstr(cand)
    return "0.8.21"  # default

def fallback_candidates(base_version: str, lo_hi_legacy: Tuple[Tuple[int,int,int],Tuple[int,int,int],bool]) -> List[str]:
    lo,hi,legacy = lo_hi_legacy
    cands = [base_version]
    if hi != (99,99,99):
        best = (hi[0],hi[1],max(0,hi[2]-1))
        if vstr(best) not in cands: cands.append(vstr(best))
    if legacy:
        if hi == (99,99,99) or (0,5,17) < hi and (0,5,0) >= lo:
            if "0.5.17" not in cands: cands.append("0.5.17")
    M,m,_ = vtup(base_version)
    if (M,m) in LATEST_PATCH:
        for p in range(LATEST_PATCH[(M,m)], -1, -1):
            vv = f"{M}.{m}.{p}"
            if vv not in cands: cands.append(vv)
    for M2,m2 in MINOR_ORDER:
        vv = vstr(latest_in_minor(M2,m2))
        if vv not in cands: cands.append(vv)
    return cands

def compile_project_stage1(project_dir: Path, csv_version: str):
    std = _load_standard_input(project_dir)
    sources = None if std else _collect_sources_from_dir(project_dir)
    n_sources = (len(std["sources"]) if std and "sources" in std else (len(sources) if sources else 0))
    src_for_infer = {k:(v.get("content") if isinstance(v, dict) else str(v)) for k,v in (std["sources"] if std else (sources or {})).items()} if (std or sources) else {}

    lo,hi,legacy = scan_pragmas(src_for_infer)
    initial = pick_initial_version(csv_version, src_for_infer)
    cands = fallback_candidates(initial, (lo,hi,legacy))

    spdx_downgraded = False
    tried = set()
    for ver in cands:
        if ver in tried: continue
        tried.add(ver)

        ok, msg, mode = _compile_once(ver, std, sources)
        if ok:
            tag = "initial" if ver == initial else ("pragma_best" if hi!=(99,99,99) and ver==vstr((hi[0],hi[1],max(0,hi[2]-1))) else ("spdx_downgrade" if spdx_downgraded else "fallback"))
            return True, f"Compiled OK ({tag} v{ver}, mode={mode})", ver, n_sources, mode

        if _looks_version_mismatch(msg) and hi != (99,99,99):
            pb = vstr((hi[0],hi[1],max(0,hi[2]-1)))
            if pb not in tried:
                ok2, msg2, mode2 = _compile_once(pb, std, sources)
                tried.add(pb)
                if ok2:
                    return True, f"Compiled OK (pragma_best v{pb}, mode={mode2})", pb, n_sources, mode2
                msg = msg + "\n---\n" + msg2

        if _looks_spdx_error(msg):
            for M2,m2 in MINOR_ORDER:
                vv = vstr(latest_in_minor(M2,m2))
                if vv in tried: continue
                spdx_downgraded = True
                ok3, msg3, mode3 = _compile_once(vv, std, sources)
                tried.add(vv)
                if ok3:
                    return True, f"Compiled OK (spdx_downgrade v{vv}, mode={mode3})", vv, n_sources, mode3
                msg = msg + "\n---\n" + msg3

        if _looks_contract_vs_address0(msg):
            return False, "Requires source change: contract vs address(0) comparison under 0.5+.", initial, n_sources, "none"

    return False, (msg if isinstance(msg,str) else "Compilation failed (no further detail)"), initial, n_sources, "none"

# ---------------- Vendor/remappings-aware (compile_files) ----------------
def read_remappings_txt(project_dir: Path) -> List[str]:
    out=[]
    f = project_dir / "remappings.txt"
    if not f.exists(): return out
    try:
        for line in f.read_text().splitlines():
            s=line.strip()
            if not s or s.startswith("#") or "=" not in s: continue
            k,v = s.split("=",1)
            k = k.strip(); v = v.strip().rstrip("/")
            abs_v = (project_dir / v).resolve()
            if abs_v.exists(): out.append(f"{k}={str(abs_v)}/")
    except Exception:
        pass
    return out

KNOWN_VENDOR_NAMES = [
    "openzeppelin-contracts","openzeppelin-contracts-upgradeable",
    "erc721a","solmate","forge-std","ds-test","chainlink",
    "uniswap-v3-core","uniswap-v3-periphery","uniswap-v2-core","uniswap-v2-periphery",
]

def scan_vendor_dirs(project_dir: Path, max_depth: int = 3) -> Dict[str, Path]:
    roots = {}
    candidates = []
    for base in ["lib", "node_modules", "vendor"]:
        root = project_dir / base
        if root.exists() and root.is_dir():
            candidates.append(root)
    q = [(c,0) for c in candidates]
    seen = set(candidates)
    while q:
        cur,d = q.pop(0)
        if d>max_depth: continue
        try:
            for child in cur.iterdir():
                if child.is_dir() and child not in seen:
                    seen.add(child); q.append((child,d+1))
        except Exception:
            pass
    for path in seen:
        name = path.name.lower()
        if any(k in name for k in KNOWN_VENDOR_NAMES):
            if "openzeppelin" in name: roots["@openzeppelin/"]=path
            if "erc721a" in name:      roots["erc721a/"]=path
            if "solmate" in name:      roots["solmate/"]=path
            if "forge-std" in name:    roots["forge-std/"]=path
            if "ds-test" in name:      roots["ds-test/"]=path
    return roots

def build_import_remappings(project_dir: Path) -> List[str]:
    remaps = read_remappings_txt(project_dir)
    auto = scan_vendor_dirs(project_dir)
    existing = {r.split("=",1)[0] for r in remaps}
    for k,p in auto.items():
        if k not in existing:
            remaps.append(f"{k}={str(p.resolve())}/")
    remaps.append(f"=/={str(project_dir.resolve())}/")  # neutral root mapping trick
    # dedup
    out,seen=[],set()
    for r in remaps:
        if r not in seen:
            out.append(r); seen.add(r)
    return out

def compile_with_remappings(version: str, project_dir: Path, files: List[Path]) -> Tuple[bool,str]:
    err = _ensure_solc(version)
    if err: return False, err
    try:
        remaps = build_import_remappings(project_dir)
        _ = compile_files(
            [str(p) for p in files],
            import_remappings=remaps,
            base_path=str(project_dir.resolve()),
            allow_paths=str(project_dir.resolve()),
            optimize=True, optimize_runs=200
        )
        return True, "Compiled OK (compile_files with remappings)"
    except Exception as e:
        return False, _err_text(e)

def stage2_vendor_retry(df: pd.DataFrame) -> pd.DataFrame:
    mask = df.get("recompile_ok").astype(str).str.lower() != "true"
    subset = df[mask]
    it = subset.iterrows()
    if _HAS_TQDM: it = tqdm(it, total=len(subset), desc="Stage2: vendor/remap retry", unit="row")

    for i,row in it:
        addr = str(row["contract_address"]).strip().lower()
        proj_dir = Path(row["project_dir"]) if isinstance(row.get("project_dir"), str) and row.get("project_dir") else (PROJECTS_DIR / addr)
        t0 = time.perf_counter()
        if not proj_dir.exists():
            df.at[i,"recompile_ok"]=False; df.at[i,"recompile_message"]="Project directory not found"; df.at[i,"elapsed_s"]=round(time.perf_counter()-t0,3)
            continue
        files = sorted(proj_dir.rglob("*.sol"))
        if not files:
            df.at[i,"recompile_ok"]=False; df.at[i,"recompile_message"]="No .sol sources found"; df.at[i,"elapsed_s"]=round(time.perf_counter()-t0,3)
            continue

        # try hint version if present, else MAX_SOLC
        prev_msg = str(row.get("recompile_message") or "")
        hinted = _extract_semver_from_error(prev_msg) or MAX_SOLC
        used = ""
        last = ""
        for ver in [hinted, MAX_SOLC]:
            ok,msg = compile_with_remappings(ver, proj_dir, files)
            if ok:
                df.at[i,"recompile_ok"]=True; df.at[i,"recompile_message"]=f"Compiled OK (remapped v{ver})"
                df.at[i,"used_version"]=ver; df.at[i,"mode"]="entry_files"; used=ver; break
            last = msg
            # classify truly unfixable code issues to stop chasing
            if _looks_override_list_error(last):
                df.at[i,"recompile_message"]="Requires source change: multiple-inheritance override must list all bases (override(...))."; break
            if _looks_data_location_mismatch(last):
                df.at[i,"recompile_message"]="Requires source change: data location mismatch on override."; break

        if not bool(df.at[i,"recompile_ok"]):
            df.at[i,"recompile_ok"]=False
            df.at[i,"used_version"]=used or (row.get("used_version") or "")
            df.at[i,"mode"]=row.get("mode") or "none"
            df.at[i,"recompile_message"]=df.at[i,"recompile_message"] or (last or "Compilation failed")

        df.at[i,"n_sources"]=len(files)
        df.at[i,"elapsed_s"]=round(time.perf_counter()-t0,3)

    return df

# ---------------- Stage 3: aggressive retry ----------------
def settings_profiles_for(v: str) -> List[dict]:
    base = {"optimizer":{"enabled":True,"runs":200}}
    profiles = [base]
    M,m,_ = vtup(v)
    if (M,m) in {(0,8),(0,7)}:
        for evm in ["istanbul","berlin","london","paris"]:
            profiles.append({"optimizer":{"enabled":True,"runs":200},"evmVersion":evm})
    elif (M,m) in {(0,6),(0,5),(0,4)}:
        for evm in ["byzantium","petersburg","istanbul"]:
            profiles.append({"optimizer":{"enabled":True,"runs":200},"evmVersion":evm})
    profiles.append({"optimizer":{"enabled":False}})
    return profiles

def minors_between(lo: Tuple[int,int,int], hi: Tuple[int,int,int]) -> List[Tuple[int,int]]:
    allowed=[]
    lo_minor = lo[1] if lo[0]==0 else 99
    hi_minor = (hi[1]-1) if (hi[0]==0 and hi[1]>0) else (hi[1] if hi[0]==0 else -1)
    if lo_minor<=hi_minor and hi_minor<=8:
        for m in range(hi_minor, lo_minor-1, -1):
            if (0,m) in LATEST_PATCH:
                allowed.append((0,m))
    return allowed

def build_candidates(prev_used: str, prev_msg: str, src_map: Dict[str,str]) -> List[str]:
    lo,hi,legacy = scan_pragmas(src_map)
    floor = _feature_floor(src_map)
    cands: List[str] = []
    seen=set()
    def add(v):
        v2=_clamp_max(v)
        if vtup(v2) < vtup(floor): v2 = floor
        if v2 not in seen: seen.add(v2); cands.append(v2)
    hinted = _extract_semver_from_error(prev_msg or "")
    if hinted: add(hinted)
    if prev_used: add(prev_used)
    if hi != (99,99,99):
        best = vstr((hi[0],hi[1],max(0,hi[2]-1))); add(best)
    add(floor)
    allowed_minors = minors_between(lo,hi) if hi!=(99,99,99) else []
    for (M,m) in allowed_minors:
        add(vstr(latest_in_minor(M,m)))
        if m==8:
            for p in [30,29,28,27,26,25,24,23,22,21,20,19,18,17,16,15]:
                add(f"0.8.{p}")
    for (M,m) in MINOR_ORDER:
        add(vstr(latest_in_minor(M,m)))
    return cands[:60]

def _compile_once_with_profiles(version: str, std: Optional[dict], sources: Optional[Dict[str,str]], settings_profiles: List[dict]) -> Tuple[bool,str,str]:
    err = _ensure_solc(version)
    if err: return False, err, "init"
    last_msg=""
    for st in settings_profiles:
        if std and isinstance(std.get("sources"), dict) and std["sources"]:
            try:
                payload = {"language":"Solidity",
                           "sources":{k:({"content": (v.get("content") if isinstance(v, dict) else str(v))})
                                      for k,v in std["sources"].items()},
                           "settings": st}
                _ = _compile_standard_resilient(payload)
                return True, "Compiled OK", "standard"
            except Exception as e:
                last_msg = f"SolcError (v{version}, mode=standard)\n{_err_text(e)}"
        elif sources:
            try:
                payload={"language":"Solidity",
                         "sources":{k:{"content":v} for k,v in sources.items()},
                         "settings": st}
                _ = _compile_standard_resilient(payload)
                return True, "Compiled OK", "all_files"
            except Exception as e_all:
                try:
                    main_rel = max(sources.items(), key=lambda kv: len(kv[1]))[0]
                    _ = compile_source(sources[main_rel])
                    return True, f"Compiled OK (single file: {main_rel})", "single_file"
                except Exception as e_single:
                    last_msg = f"SolcError (v{version}, mode=all_files)\n{_err_text(e_all)}\n---\nSolcError (v{version}, mode=single_file)\n{_err_text(e_single)}"
        else:
            return False, "No .sol sources found", "none"
    return False, last_msg or "Compilation failed", "none"

def stage3_aggressive_retry(df: pd.DataFrame) -> pd.DataFrame:
    mask = df.get("recompile_ok").astype(str).str.lower() != "true"
    it = df[mask].iterrows()
    if _HAS_TQDM: it = tqdm(it, total=mask.sum(), desc="Stage3: aggressive retry", unit="row")
    for i,row in it:
        addr = str(row["contract_address"]).strip().lower()
        proj_dir = Path(row["project_dir"]) if isinstance(row.get("project_dir"), str) and row.get("project_dir") else (PROJECTS_DIR / addr)
        t0 = time.perf_counter()
        if not proj_dir.exists():
            df.at[i,"recompile_ok"]=False; df.at[i,"recompile_message"]="Project directory not found"; df.at[i,"elapsed_s"]=round(time.perf_counter()-t0,3); continue
        std = _load_standard_input(proj_dir)
        if std and isinstance(std.get("sources"), dict) and std["sources"]:
            src_map = {k:(v.get("content") if isinstance(v, dict) else str(v)) for k,v in std["sources"].items()}
            sources_for_all = None
        else:
            src_map = _collect_sources_from_dir(proj_dir)
            sources_for_all = src_map
        if not src_map:
            df.at[i,"recompile_ok"]=False; df.at[i,"recompile_message"]="No .sol sources found"; df.at[i,"elapsed_s"]=round(time.perf_counter()-t0,3); continue

        prev_used = str(row.get("used_version") or "")
        prev_msg  = str(row.get("recompile_message") or "")
        candidates = build_candidates(prev_used, prev_msg, src_map)
        last=""; used=""; mode="none"; ok=False
        for ver in candidates:
            profiles = settings_profiles_for(ver)
            ok,msg,mode = _compile_once_with_profiles(ver, std, sources_for_all, profiles)
            if ok:
                used = ver
                df.at[i,"recompile_ok"]=True; df.at[i,"recompile_message"]=f"Compiled OK (aggressive v{ver}, mode={mode})"; df.at[i,"used_version"]=ver; df.at[i,"mode"]=mode
                break
            last = msg
            if _looks_override_list_error(last):
                df.at[i,"recompile_ok"]=False; df.at[i,"recompile_message"]="Requires source change: override must list all bases."; break
            if _looks_data_location_mismatch(last):
                df.at[i,"recompile_ok"]=False; df.at[i,"recompile_message"]="Requires source change: data location mismatch on override."; break
        if not ok and not df.at[i,"recompile_message"]:
            df.at[i,"recompile_ok"]=False; df.at[i,"recompile_message"]=last or "Compilation failed"; df.at[i,"used_version"]=used or prev_used; df.at[i,"mode"]=mode
        df.at[i,"n_sources"]=len(src_map); df.at[i,"elapsed_s"]=round(time.perf_counter()-t0,3)
    return df

# ---------------- Stage 4: entry-file strategy ----------------
def _file_pragma_max_tuple(text: str) -> Tuple[int,int,int]:
    lo,hi = (0,0,0),(99,99,99)
    for pm in PRAGMA.finditer(text or ""):
        a,b = parse_pragma_expr(pm.group(1))
        lo = a if a>lo else lo
        hi = b if b<hi else hi
    if hi == (99,99,99): return (-1,-1,-1)
    return (hi[0],hi[1],max(0,hi[2]-1))

def _entry_candidates(project_dir: Path, limit=3) -> List[Path]:
    files = list(project_dir.rglob("*.sol"))
    scored=[]
    for p in files:
        try: t=p.read_text()
        except Exception: t=""
        pragma_t=_file_pragma_max_tuple(t)
        contracts=len(re.findall(r"\bcontract\b", t))
        score=(pragma_t, contracts, len(t))
        scored.append((score,p))
    scored.sort(reverse=True)
    return [p for _,p in scored[:limit]] if scored else []

def stage4_entry_retry(df: pd.DataFrame) -> pd.DataFrame:
    mask = df.get("recompile_ok").astype(str).str.lower() != "true"
    it = df[mask].iterrows()
    if _HAS_TQDM: it = tqdm(it, total=mask.sum(), desc="Stage4: entry-file retry", unit="row")
    for i,row in it:
        addr = str(row["contract_address"]).strip().lower()
        proj_dir = Path(row["project_dir"]) if isinstance(row.get("project_dir"), str) and row.get("project_dir") else (PROJECTS_DIR / addr)
        t0 = time.perf_counter()
        if not proj_dir.exists():
            df.at[i,"recompile_ok"]=False; df.at[i,"recompile_message"]="Project directory not found"; df.at[i,"elapsed_s"]=round(time.perf_counter()-t0,3); continue
        entries = _entry_candidates(proj_dir, limit=3)
        if not entries:
            df.at[i,"recompile_ok"]=False; df.at[i,"recompile_message"]="No .sol sources found"; df.at[i,"elapsed_s"]=round(time.perf_counter()-t0,3); continue

        # seed versions: hinted -> MAX_SOLC -> previous used
        prev_msg = str(row.get("recompile_message") or "")
        prev_used = str(row.get("used_version") or "")
        seeds = []
        hinted = _extract_semver_from_error(prev_msg)
        if hinted: seeds.append(hinted)
        seeds.append(MAX_SOLC)
        if prev_used and prev_used not in seeds: seeds.append(prev_used)

        last=""; used=""; ok=False
        for ver in seeds:
            err = _ensure_solc(ver)
            if err: last=err; continue
            for entry in entries:
                try:
                    _ = compile_files([str(entry)], base_path=str(proj_dir), allow_paths=str(proj_dir), optimize=True, optimize_runs=200)
                    df.at[i,"recompile_ok"]=True; df.at[i,"recompile_message"]=f"Compiled OK (entry:{entry.name})"; df.at[i,"used_version"]=ver; df.at[i,"mode"]="entry_files"
                    ok=True; used=ver; break
                except Exception as e:
                    last = f"SolcError (v{ver}, entry={entry.name})\n{_err_text(e)}"
            if ok: break

        if not ok:
            df.at[i,"recompile_ok"]=False; df.at[i,"recompile_message"]=last or "Compilation failed"; df.at[i,"used_version"]=used or prev_used; df.at[i,"mode"]=row.get("mode") or "none"

        df.at[i,"n_sources"]=sum(1 for _ in proj_dir.rglob("*.sol"))
        df.at[i,"elapsed_s"]=round(time.perf_counter()-t0,3)

    return df

# ---------------- Stage 1 driver (build OUT_CSV fresh) ----------------
def stage1_full_pass():
    df_in = pd.read_csv(INPUT_CSV)
    todo = df_in.copy()

    rows=[]
    it = range(len(todo))
    if _HAS_TQDM:
        it = tqdm(it, desc="Stage1: smart full pass", unit="addr")

    for idx in it:
        row = todo.iloc[idx]
        addr = str(row["contract_address"]).strip().lower()
        proj = PROJECTS_DIR / addr
        t0 = time.perf_counter()

        if not proj.exists():
            rows.append({
                "contract_address": addr, "project_dir": str(proj),
                "recompile_ok": False, "recompile_message": "Project directory not found",
                "used_version": "", "n_sources": 0, "mode": "none",
                "elapsed_s": round(time.perf_counter() - t0, 3),
                "original_idx": row.get("original_idx"), "label": row.get("label"),
            })
            continue

        csv_ver = str(row.get("compiler_version") or "").strip()
        ok, msg, used, nsrc, mode = compile_project_stage1(proj, csv_ver)

        rows.append({
            "contract_address": addr, "project_dir": str(proj),
            "recompile_ok": bool(ok), "recompile_message": str(msg),
            "used_version": used, "n_sources": int(nsrc), "mode": mode,
            "elapsed_s": round(time.perf_counter() - t0, 3),
            "original_idx": row.get("original_idx"), "label": row.get("label"),
        })
        if SLEEP_BETWEEN>0: time.sleep(SLEEP_BETWEEN)

    out = pd.DataFrame(rows)
    out.to_csv(OUT_CSV, index=False)
    print(f"[Stage1] wrote {OUT_CSV} ({len(out)} rows)")
    return out

# ---------------- Main orchestrator ----------------
def main():
    if RUN_STAGE_1:
        df = stage1_full_pass()
    else:
        if not Path(OUT_CSV).exists():
            raise SystemExit(f"{OUT_CSV} not found. Run stage 1 or point OUT_CSV to an existing results file.")
        df = pd.read_csv(OUT_CSV)

    # Stage 2
    if RUN_STAGE_2:
        before = df.get("recompile_ok").astype(str).str.lower().eq("true").sum()
        df = stage2_vendor_retry(df)
        df.to_csv(OUT_CSV, index=False)
        after = df.get("recompile_ok").astype(str).str.lower().eq("true").sum()
        print(f"[Stage2] successes: +{after - before} (total OK={after})")

    # Stage 3
    if RUN_STAGE_3:
        before = df.get("recompile_ok").astype(str).str.lower().eq("true").sum()
        df = stage3_aggressive_retry(df)
        df.to_csv(OUT_CSV, index=False)
        after = df.get("recompile_ok").astype(str).str.lower().eq("true").sum()
        print(f"[Stage3] successes: +{after - before} (total OK={after})")

    # Stage 4
    if RUN_STAGE_4:
        before = df.get("recompile_ok").astype(str).str.lower().eq("true").sum()
        df = stage4_entry_retry(df)
        df.to_csv(OUT_CSV, index=False)
        after = df.get("recompile_ok").astype(str).str.lower().eq("true").sum()
        print(f"[Stage4] successes: +{after - before} (total OK={after})")

    # final summary
    ok_count = df.get("recompile_ok").astype(str).str.lower().eq("true").sum()
    fail_count = len(df) - ok_count
    print(f"[DONE] OK={ok_count}  FAIL={fail_count}  → {OUT_CSV}")

if __name__ == "__main__":
    main()


## Compilability of flames-20k results

In [ ]:
# === Cell 1: Setup & mapping ===
# REQS: pip install pandas py-solc-x tqdm
import os, re, json, time, shutil
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import pandas as pd

# ---- Paths ----
# SYN_CSV   = "synthesized_invariants-20k.csv"  # must have: original_idx, label, prediction
SYN_CSV   = "synthesized_invariants-codellama.csv"  # must have: original_idx, label, prediction
MATCH_CSV = "label_require_match.csv"         # must map (original_idx,label) -> project/file/line
PROJECTS_DIR = Path("contracts")              # base for projects when 'project_dir' is missing

# ---- Column name fallbacks (edit if yours differ) ----
COL_ORIG   = "original_idx"
COL_LABEL  = "label"
COL_PRED   = "prediction"

# where to patch (any of these present in MATCH_CSV)
COL_MATCH_FOUND = "match_found"
COL_PROJ_DIR    = "project_dir"               # preferred absolute/relative project root
COL_ADDRESS     = "contract_address"          # fallback to PROJECTS_DIR/<address>
COL_REL_PATHS   = ["relative_path","file_path","req_relative_path","path"]
COL_LINE_NO     = ["line_no","req_line_no","lineno"]

# ---- Safety: backup suffix for temporary patching ----
BACKUP_SUFFIX = ".bak.__inject__"

# Load input CSVs
df_syn   = pd.read_csv(SYN_CSV)
df_match = pd.read_csv(MATCH_CSV)

# Minimal validation
need_syn = {COL_ORIG, COL_LABEL, COL_PRED}
if not need_syn.issubset(df_syn.columns):
    raise SystemExit(f"{SYN_CSV} must contain columns: {sorted(need_syn)}")

# Normalize join keys
df_syn[COL_ORIG]  = pd.to_numeric(df_syn[COL_ORIG], errors="coerce").astype("Int64")
df_syn[COL_LABEL] = df_syn[COL_LABEL].astype(str)

# Build flexible accessors for match CSV
def _first_col(cols: List[str]) -> Optional[str]:
    for c in cols:
        if c in df_match.columns:
            return c
    return None

REL_COL  = _first_col(COL_REL_PATHS)
LINE_COL = _first_col(COL_LINE_NO)

if REL_COL is None or LINE_COL is None:
    raise SystemExit(
        f"{MATCH_CSV} must contain a relative path and a line number column. "
        f"Tried path candidates {COL_REL_PATHS}, line candidates {COL_LINE_NO}."
    )

# Normalize keys in match CSV
df_match[COL_ORIG]  = pd.to_numeric(df_match[COL_ORIG], errors="coerce").astype("Int64")
df_match[COL_LABEL] = df_match[COL_LABEL].astype(str)
if COL_MATCH_FOUND in df_match.columns:
    df_match = df_match[df_match[COL_MATCH_FOUND].astype(str).str.lower() == "true"].copy()

# Keep only what we need to locate the file to patch
cols_keep = [COL_ORIG, COL_LABEL, REL_COL, LINE_COL]
if COL_PROJ_DIR in df_match.columns: cols_keep.append(COL_PROJ_DIR)
if COL_ADDRESS  in df_match.columns: cols_keep.append(COL_ADDRESS)
df_loc = df_match[cols_keep].drop_duplicates(subset=[COL_ORIG, COL_LABEL], keep="first").copy()

# Build index for fast lookup
match_idx = df_loc.set_index([COL_ORIG, COL_LABEL])
print(f"[info] loaded syn={len(df_syn)}  matches (usable)={len(match_idx)}")

In [ ]:
df_syn.iloc[0]

In [ ]:
# === Cell 2: Require parsing / patching helpers ===
def _normalize_pred(pred: str) -> str:
    """Use only the inner expression if someone returned 'require(<expr>);'."""
    s = (pred or "").strip()
    s = re.sub(r";\s*$", "", s)  # drop trailing ';'
    m = re.search(r"require\s*\(\s*(.*?)\s*\)\s*$", s, flags=re.S)
    return (m.group(1).strip() if m else s)

def _find_require_span_around_line(text: str, target_line_one_based: int) -> Tuple[int,int]:
    """
    Return (open_paren_idx, close_paren_idx) for the first 'require(' that starts
    at or just after the target line. Robust to multi-line requires and strings.
    Raises ValueError if not found.
    """
    # Map line -> absolute char offset
    lines = text.splitlines(keepends=True)
    if target_line_one_based < 1 or target_line_one_based > len(lines):
        raise ValueError("line out of range")
    start_abs = sum(len(l) for l in lines[:target_line_one_based-1])
    # search from a bit before the target line to catch broken formatting
    search_start = max(0, start_abs - 200)
    m = re.search(r"\brequire\s*\(", text[search_start:], flags=re.M)
    if not m:
        raise ValueError("require( not found near target line")
    open_idx = search_start + m.end() - 1  # points to '('

    # scan forward to find matching ')'
    i = open_idx
    depth = 0
    in_str = None   # '"'/ "'"
    escaped = False
    while i < len(text):
        ch = text[i]
        if in_str:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == in_str:
                in_str = None
        else:
            if ch in ("'", '"'):
                in_str = ch
            elif ch == "(":
                depth += 1
            elif ch == ")":
                depth -= 1
                if depth == 0:
                    return (open_idx, i)
        i += 1
    raise ValueError("closing ')' for require(...) not found")

def _split_condition_and_msg(inside: str) -> Tuple[str, Optional[str]]:
    """
    Split 'cond, "msg"' at top-level comma (ignore commas inside strings/parentheses).
    Return (cond, msg_or_None).
    """
    depth = 0
    in_str = None
    escaped = False
    for i,ch in enumerate(inside):
        if in_str:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == in_str:
                in_str = None
        else:
            if ch in ("'", '"'):
                in_str = ch
            elif ch == "(":
                depth += 1
            elif ch == ")":
                depth -= 1
            elif ch == "," and depth == 0:
                cond = inside[:i].strip()
                msg  = inside[i+1:].strip()
                return cond, msg if msg else None
    return inside.strip(), None

def patch_require_in_file(file_path: Path, line_no: int, new_cond_raw: str) -> Tuple[str, str]:
    """
    Patch the require at/after given line: replace first arg with new_cond_raw.
    Returns (old_inside, new_inside).
    Raises on failure; caller should handle backup/restore.
    """
    text = file_path.read_text(encoding="utf-8", errors="ignore")
    open_idx, close_idx = _find_require_span_around_line(text, int(line_no))
    inside = text[open_idx+1:close_idx]

    cond_old, msg_opt = _split_condition_and_msg(inside)
    new_cond = _normalize_pred(new_cond_raw)

    if msg_opt is not None:
        new_inside = f"{new_cond}, {msg_opt}"
    else:
        new_inside = new_cond

    new_text = text[:open_idx+1] + new_inside + text[close_idx:]
    file_path.write_text(new_text, encoding="utf-8")
    return inside, new_inside


In [ ]:
# === Cell 3: Robust compile helpers (mirrors your pipeline) ===
from solcx import (
    install_solc, set_solc_version, get_installed_solc_versions,
    compile_standard, compile_source, compile_files
)
from solcx.exceptions import SolcError

try:
    from tqdm.auto import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False

MAX_SOLC    = "0.8.30"
SEMVER3     = re.compile(r"(\d+)\.(\d+)\.(\d+)")
PRAGMA      = re.compile(r"pragma\s+solidity\s+([^;]+);", re.IGNORECASE)
OP_VER      = re.compile(r"([~^]|>=|<=|>|<)?\s*(\d+)\.(\d+)\.(\d+)")
ERR_PRAGMA_SEMVER = re.compile(r"pragma\s+solidity[^0-9]*?(\d+\.\d+\.\d+)", re.IGNORECASE)

LATEST_PATCH = {(0,4):26,(0,5):17,(0,6):12,(0,7):6,(0,8):30}
MINOR_ORDER  = [(0,8),(0,7),(0,6),(0,5),(0,4)]

def vtup(s: str):
    m = SEMVER3.search(s or ""); 
    return tuple(map(int, m.groups())) if m else (0,0,0)
def vstr(t): return f"{t[0]}.{t[1]}.{t[2]}"
def latest_in_minor(M,m): return (M,m, LATEST_PATCH.get((M,m), 0))
def _clamp_max(v: str, max_v: str = MAX_SOLC) -> str:
    return max_v if vtup(v) > vtup(max_v) else v

def _ensure_solc(version: str) -> Optional[str]:
    version = _clamp_max(version)
    try:
        installed = {str(v) for v in get_installed_solc_versions()}
        if version not in installed: install_solc(version)
        set_solc_version(version)
        return None
    except Exception as e:
        return f"Failed to set solc {version}: {e}"

def _err_text(e: Exception) -> str:
    if isinstance(e, SolcError):
        stderr = getattr(e, "stderr_data", None) or getattr(e, "stderr", None) or ""
        stdout = getattr(e, "stdout_data", None) or getattr(e, "stdout", None) or ""
        tail = lambda s: s[-1400:] if isinstance(s,str) else ""
        return f"--- stderr (tail) ---\n{tail(stderr)}\n--- stdout (tail) ---\n{tail(stdout)}"
    return str(e)

def parse_pragma_expr(expr: str):
    lo,hi=(0,0,0),(99,99,99)
    toks=OP_VER.findall(expr or "")
    if not toks: return lo,hi
    vmax=lambda a,b: a if a>b else b
    vmin=lambda a,b: a if a<b else b
    for op,a,b,c in toks:
        vt=(int(a),int(b),int(c))
        if op in ("",None): lo=vmax(lo,vt); hi=vmin(hi,(vt[0],vt[1],vt[2]+1))
        elif op in ("^","~"): lo=vmax(lo,vt); hi=vmin(hi,(vt[0],vt[1]+1,0))
        elif op==">=": lo=vmax(lo,vt)
        elif op==">":  lo=vmax(lo,(vt[0],vt[1],vt[2]+1))
        elif op=="<=": hi=vmin(hi,(vt[0],vt[1],vt[2]+1))
        elif op=="<":  hi=vmin(hi,vt)
    return lo,hi

def _collect_sources_from_dir(project_dir: Path) -> Dict[str,str]:
    out={}
    for p in project_dir.rglob("*.sol"):
        try: out[p.relative_to(project_dir).as_posix()] = p.read_text()
        except Exception: out[p.relative_to(project_dir).as_posix()] = ""
    return out

def _load_standard_input(project_dir: Path) -> Optional[dict]:
    f = project_dir / "standard_input.json"
    if not f.exists(): return None
    try: return json.loads(f.read_text())
    except Exception: return None

def _compile_standard_resilient(payload: dict):
    try:
        return compile_standard(payload)
    except SolcError as e1:
        txt = _err_text(e1)
        if "Invalid EVM version requested" in txt:
            st = dict(payload.get("settings") or {}); st.pop("evmVersion", None)
            p2 = dict(payload); p2["settings"]=st
            return compile_standard(p2)
        raise

def _compile_once(version: str, std: Optional[dict], sources: Optional[Dict[str,str]]) -> Tuple[bool,str,str]:
    err=_ensure_solc(version)
    if err: return False, err, "init"
    if std and isinstance(std.get("sources"), dict) and std["sources"]:
        try:
            settings = dict(std.get("settings") or {})
            if "optimizer" not in settings: settings["optimizer"]={"enabled":True,"runs":200}
            payload={"language":"Solidity",
                     "sources":{k:({"content": (v.get("content") if isinstance(v, dict) else str(v))})
                                for k,v in std["sources"].items()},
                     "settings":settings}
            _=_compile_standard_resilient(payload)
            return True, "Compiled OK", "standard"
        except Exception as e:
            return False, f"SolcError (v{version}, mode=standard)\n{_err_text(e)}", "standard"
    if sources:
        try:
            payload={"language":"Solidity",
                     "sources":{k:{"content":v} for k,v in sources.items()},
                     "settings":{"optimizer":{"enabled":True,"runs":200}}}
            _=_compile_standard_resilient(payload)
            return True,"Compiled OK","all_files"
        except Exception as e_all:
            try:
                main_rel = max(sources.items(), key=lambda kv: len(kv[1]))[0]
                _ = compile_source(sources[main_rel])
                return True, f"Compiled OK (single file: {main_rel})", "single_file"
            except Exception as e_single:
                joined = f"SolcError (v{version}, mode=all_files)\n{_err_text(e_all)}\n---\nSolcError (v{version}, mode=single_file)\n{_err_text(e_single)}"
                return False, joined, "all_files"
    return False, "No .sol sources found", "none"

def _looks_spdx_error(msg: str) -> bool:
    return "Invalid SPDX license identifier" in (msg or "")

def _extract_semver_from_error(msg: str) -> Optional[str]:
    m = ERR_PRAGMA_SEMVER.search(msg or ""); 
    if not m: return None
    v = m.group(1);  # clamp bogus 0.9.x
    return MAX_SOLC if vtup(v)[1] >= 9 else v

def read_remappings_txt(project_dir: Path) -> List[str]:
    out=[]; f=project_dir/"remappings.txt"
    if not f.exists(): return out
    try:
        for line in f.read_text().splitlines():
            s=line.strip()
            if not s or s.startswith("#") or "=" not in s: continue
            k,v=s.split("=",1); k=k.strip(); v=v.strip().rstrip("/")
            abs_v=(project_dir / v).resolve()
            if abs_v.exists(): out.append(f"{k}={str(abs_v)}/")
    except Exception: pass
    return out

KNOWN_VENDOR_NAMES = [
    "openzeppelin-contracts","openzeppelin-contracts-upgradeable",
    "erc721a","solmate","forge-std","ds-test","chainlink",
    "uniswap-v3-core","uniswap-v3-periphery","uniswap-v2-core","uniswap-v2-periphery",
]

def scan_vendor_dirs(project_dir: Path, max_depth: int = 3) -> Dict[str, Path]:
    roots={}; candidates=[]
    for base in ["lib","node_modules","vendor"]:
        root=project_dir/base
        if root.exists() and root.is_dir(): candidates.append(root)
    q=[(c,0) for c in candidates]; seen=set(candidates)
    while q:
        cur,d=q.pop(0)
        if d>max_depth: continue
        try:
            for ch in cur.iterdir():
                if ch.is_dir() and ch not in seen:
                    seen.add(ch); q.append((ch,d+1))
        except Exception: pass
    for path in seen:
        name=path.name.lower()
        if any(k in name for k in KNOWN_VENDOR_NAMES):
            if "openzeppelin" in name: roots["@openzeppelin/"]=path
            if "erc721a" in name:      roots["erc721a/"]=path
            if "solmate" in name:      roots["solmate/"]=path
            if "forge-std" in name:    roots["forge-std/"]=path
            if "ds-test" in name:      roots["ds-test/"]=path
    return roots

def build_import_remappings(project_dir: Path) -> List[str]:
    remaps=read_remappings_txt(project_dir)
    auto=scan_vendor_dirs(project_dir)
    existing={r.split("=",1)[0] for r in remaps}
    for k,p in auto.items():
        if k not in existing: remaps.append(f"{k}={str(p.resolve())}/")
    remaps.append(f"=/={str(project_dir.resolve())}/")  # neutral root mapping trick
    out,seen=[],set()
    for r in remaps:
        if r not in seen:
            out.append(r); seen.add(r)
    return out

def compile_with_remappings(version: str, project_dir: Path, files: List[Path]) -> Tuple[bool,str]:
    err=_ensure_solc(version)
    if err: return False, err
    try:
        remaps=build_import_remappings(project_dir)
        _=compile_files(
            [str(p) for p in files],
            import_remappings=remaps,
            base_path=str(project_dir.resolve()),
            allow_paths=str(project_dir.resolve()),
            optimize=True, optimize_runs=200
        )
        return True, "Compiled OK (compile_files with remappings)"
    except Exception as e:
        return False, _err_text(e)

def settings_profiles_for(v: str) -> List[dict]:
    base={"optimizer":{"enabled":True,"runs":200}}
    profiles=[base]
    M,m,_=vtup(v)
    if (M,m) in {(0,8),(0,7)}:
        for evm in ["istanbul","berlin","london","paris"]:
            profiles.append({"optimizer":{"enabled":True,"runs":200},"evmVersion":evm})
    elif (M,m) in {(0,6),(0,5),(0,4)}:
        for evm in ["byzantium","petersburg","istanbul"]:
            profiles.append({"optimizer":{"enabled":True,"runs":200},"evmVersion":evm})
    profiles.append({"optimizer":{"enabled":False}})
    return profiles

def robust_compile_project(project_dir: Path, csv_version_hint: str = "") -> Tuple[bool,str,str]:
    """
    Returns (ok, message, used_version)
    """
    std=_load_standard_input(project_dir)
    sources=None if std else _collect_sources_from_dir(project_dir)
    n_src = (len(std["sources"]) if std and "sources" in std else (len(sources) if sources else 0))
    if n_src == 0:
        return False, "No .sol sources found", ""

    # -------- Stage 1: standard/all/single with a small version ladder --------
    def scan_pragmas(src_map: Dict[str,str]):
        lo,hi=(0,0,0),(99,99,99)
        for content in src_map.values():
            for pm in PRAGMA.finditer(content or ""):
                a,b = parse_pragma_expr(pm.group(1))
                lo = a if a>lo else lo; hi = b if b<hi else hi
        return lo,hi
    src_for_infer = {k:(v.get("content") if isinstance(v, dict) else str(v)) for k,v in (std["sources"] if std else (sources or {})).items()}
    lo,hi=scan_pragmas(src_for_infer)
    initial = csv_version_hint.strip() if csv_version_hint else (
        (vstr((hi[0],hi[1],max(0,hi[2]-1))) if hi!=(99,99,99) else "0.8.21")
    )
    ladder = [initial]
    # + a few helpful fallbacks
    ladder += ["0.8.30","0.8.21","0.7.6","0.6.12","0.5.17","0.4.26"]
    tried=set(); last=""; used=""
    for ver in ladder:
        if ver in tried: continue
        tried.add(ver)
        ok,msg,_mode = _compile_once(ver, std, sources)
        if ok: return True, f"Compiled OK (v{ver})", ver
        last = msg
        if _looks_spdx_error(last):
            for (M2,m2) in MINOR_ORDER:
                vv=vstr(latest_in_minor(M2,m2))
                if vv in tried: continue
                tried.add(vv)
                ok2,msg2,_ = _compile_once(vv, std, sources)
                if ok2: return True, f"Compiled OK (spdx_downgrade v{vv})", vv
                last = last + "\n---\n" + msg2

        hinted = _extract_semver_from_error(last)
        if hinted and hinted not in tried:
            tried.add(hinted)
            ok3,msg3,_=_compile_once(hinted, std, sources)
            if ok3: return True, f"Compiled OK (hint v{hinted})", hinted
            last = last + "\n---\n" + msg3

    # -------- Stage 2: compile_files with remappings --------
    files = [p for p in project_dir.rglob("*.sol")]
    for ver in ["0.8.30","0.8.21","0.7.6","0.6.12","0.5.17","0.4.26"]:
        ok,msg = compile_with_remappings(ver, project_dir, files)
        if ok: return True, f"Compiled OK (remappings v{ver})", ver
        last = msg

    # -------- Stage 3: aggressive settings+evm ladder --------
    for ver in ["0.8.30","0.8.21","0.7.6","0.6.12","0.5.17","0.4.26"]:
        err=_ensure_solc(ver)
        if err: last=err; continue
        std_map = None if not std else {k: (v.get("content") if isinstance(v, dict) else str(v)) for k,v in std["sources"].items()}
        srcs    = None if std else sources
        for st in settings_profiles_for(ver):
            try:
                if std_map:
                    payload={"language":"Solidity","sources":{k:{"content":v} for k,v in std_map.items()},"settings":st}
                else:
                    payload={"language":"Solidity","sources":{k:{"content":v} for k,v in srcs.items()},"settings":st}
                _=_compile_standard_resilient(payload)
                return True, f"Compiled OK (aggressive v{ver})", ver
            except Exception as e:
                last = _err_text(e)

    # -------- Stage 4: entry-file fallback (best-effort) --------
    entries = []
    for p in project_dir.rglob("*.sol"):
        try:
            t=p.read_text()
            # heuristic: prefer files that declare contracts
            score=(len(re.findall(r"\bcontract\b", t)), len(t))
            entries.append((score,p))
        except Exception:
            continue
    entries.sort(reverse=True)
    entries=[p for _,p in entries[:3]]
    for ver in ["0.8.30","0.8.21","0.7.6","0.6.12","0.5.17","0.4.26"]:
        err=_ensure_solc(ver)
        if err: last=err; continue
        for entry in entries:
            try:
                _=compile_files([str(entry)], base_path=str(project_dir), allow_paths=str(project_dir), optimize=True, optimize_runs=200)
                return True, f"Compiled OK (entry:{entry.name} v{ver})", ver
            except Exception as e:
                last=_err_text(e)

    return False, (last or "Compilation failed"), ""


In [21]:
# === Cell 4: Inject each prediction, compile project, restore file, and update CSV ===
from tqdm.auto import tqdm
from typing import Tuple, List

def resolve_target_file(proj_dir: Path, rel_path: str) -> Tuple[Optional[Path], List[Path]]:
    """
    Given a project directory and a 'relative_path' that may accidentally
    include leading folders like 'contracts/<address>/', try several
    strategies until we find a real file on disk.
    Returns (found_path or None, attempted_candidates_list).
    """
    attempted: List[Path] = []

    # Normalize separators and remove '.', '/', '' fragments
    rp = Path(rel_path)
    parts = [p for p in rp.parts if p not in ("", "/", ".")]

    # 1) Progressive head-trim (drop leading segments until it exists)
    for i in range(len(parts)):
        cand = proj_dir.joinpath(*parts[i:])
        attempted.append(cand)
        if cand.exists():
            return cand, attempted

    # 2) Suffix match: look for a file under proj_dir whose path ends with the last 2–4 segments
    all_files = list(proj_dir.rglob("*.sol"))
    for tail_len in (4, 3, 2, 1):
        if tail_len > len(parts): 
            continue
        suffix = Path(*parts[-tail_len:]).as_posix()
        best = None
        best_len = -1
        for f in all_files:
            p = f.as_posix()
            if p.endswith("/" + suffix) or p.endswith(suffix):
                L = len(suffix)
                if L > best_len:
                    best, best_len = f, L
        if best:
            attempted.append(best)
            return best, attempted

    # 3) Basename fallback
    base = rp.name
    candidates = [f for f in all_files if f.name == base]
    if candidates:
        attempted.extend(candidates)
        return candidates[0], attempted  # best-effort

    return None, attempted


def _resolve_project_dir(row_m: pd.Series) -> Optional[Path]:
    if COL_PROJ_DIR in row_m and isinstance(row_m[COL_PROJ_DIR], str) and row_m[COL_PROJ_DIR]:
        return Path(row_m[COL_PROJ_DIR])
    if COL_ADDRESS in row_m and isinstance(row_m[COL_ADDRESS], str) and row_m[COL_ADDRESS]:
        return PROJECTS_DIR / row_m[COL_ADDRESS].strip().lower()
    return None

def _to_int(v) -> Optional[int]:
    try: return int(v)
    except Exception: return None

compiled_flags = []
compile_msgs   = []

rows_iter = tqdm(df_syn.itertuples(index=False), total=len(df_syn), desc="Inject+Compile", unit="row")
for r in rows_iter:
    oi   = getattr(r, COL_ORIG)
    lbl  = getattr(r, COL_LABEL)
    pred = getattr(r, COL_PRED)

    # Default outputs
    compiled_ok = False
    compiled_msg = "skip: no match"

    # Find the target row in match index
    key = (pd.NA if pd.isna(oi) else int(oi), str(lbl))
    if key not in match_idx.index:
        compiled_flags.append(False)
        compile_msgs.append(compiled_msg)
        continue

    row_m = match_idx.loc[key]
    # If we had duplicates we took first in .set_index; ensure we have a 1D Series
    if isinstance(row_m, pd.DataFrame):
        row_m = row_m.iloc[0]

    proj_dir = _resolve_project_dir(row_m)
    rel_path = row_m[REL_COL]
    line_no  = _to_int(row_m[LINE_COL])

    if not proj_dir or not rel_path or not line_no:
        compiled_flags.append(False)
        compile_msgs.append("skip: incomplete mapping (project/path/line)")
        continue

    target_file, tried = resolve_target_file(proj_dir, str(rel_path))
    if not target_file:
        # keep a short trace of what we tried to help debugging
        short_tried = " | ".join(map(str, tried[:4])) + (" ..." if len(tried) > 4 else "")
        compiled_flags.append(False)
        compile_msgs.append(f"skip: file not found; tried: {short_tried}")
        continue

    # Backup, patch, compile, restore
    try:
        # 1) backup
        backup_path = target_file.with_suffix(target_file.suffix + BACKUP_SUFFIX)
        if backup_path.exists():
            backup_path.unlink()
        shutil.copy2(target_file, backup_path)

        # 2) patch
        _old_inside, _new_inside = patch_require_in_file(target_file, line_no, pred)

        # 3) compile
        ok, msg, used = robust_compile_project(proj_dir, csv_version_hint="")
        compiled_ok   = bool(ok)
        compiled_msg  = msg

    except Exception as e:
        compiled_ok  = False
        compiled_msg = f"inject/compile error: {e}"

    finally:
        # 4) restore
        try:
            if backup_path.exists():
                shutil.copy2(backup_path, target_file)
                backup_path.unlink(missing_ok=True)
        except Exception:
            # even if restore fails we still record the result
            pass

    compiled_flags.append(compiled_ok)
    compile_msgs.append(compiled_msg)
    rows_iter.set_postfix(ok=compiled_ok)

# Attach results and write back to SAME CSV (add/overwrite columns)
df_syn["compiled_after_injection"]        = compiled_flags
df_syn["compile_message_after_injection"] = compile_msgs
df_syn.to_csv(SYN_CSV, index=False)
print(f"[done] Updated {SYN_CSV} with compilation results.")
print(pd.Series(compiled_flags).value_counts(dropna=False))

Inject+Compile: 100%|██████████| 5000/5000 [1:33:46<00:00,  1.13s/row, ok=0]  


[done] Updated synthesized_invariants-codellama.csv with compilation results.
True     3130
False    1870
Name: count, dtype: int64
